In [ ]:
import duckdb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.dates as mdates

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.ardl import ARDL, ardl_select_order
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    acorr_breusch_godfrey,
    het_breuschpagan,
    het_white,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.regression.rolling import RollingOLS

from functools import reduce

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Database location is read from DUCKDB_DIR (see .env.example). The DuckDB
# files are not distributed with this repository - see README "Data".
load_dotenv()
DB_PATH = Path(os.environ["DUCKDB_DIR"]).expanduser()

if not DB_PATH.is_dir():
    raise FileNotFoundError(
        f"DUCKDB_DIR points to {DB_PATH}, which does not exist. "
        "Copy .env.example to .env and set DUCKDB_DIR to your database directory."
    )

con = duckdb.connect()

for alias, filename in [
    ("country_info", "country_info.duckdb"),
    ("inflation", "inflation.duckdb"),
    ("fao", "fao.duckdb"),
    ("commodity_prices", "commodity_prices.duckdb"),
    ("eer", "effective_exchange_rates.duckdb"),
    ("exchange_rates", "exchange_rates.duckdb"),
]:
    con.execute(f"ATTACH '{DB_PATH / filename}' AS {alias} (READ_ONLY)")

In [ ]:
def normalize_quarter_dates(df):
    df["date"] = pd.PeriodIndex(df["date"], freq="Q").to_timestamp("Q")
    return df

In [ ]:
query_inflation = """
    SELECT
        s.series_code,
        v.date,
        v.index_value
    FROM       country_info.main.countries       AS c
    INNER JOIN inflation.main.cpi_series         AS s  ON s.iso2        = c.iso2
    INNER JOIN inflation.main.cpi_index_values   AS v  ON v.iso2        = s.iso2
                                                      AND v.series_code = s.series_code
    WHERE c.iso2 = 'ZA'
      AND s.series_code IN ('CPS00000', 'CPS00014', 'CPS07221', 'CPS01000')
    ORDER BY s.series_code, v.date
"""

df_inflation = con.execute(query_inflation).df()

df_inflation = (
    df_inflation.pivot(index="date", columns="series_code", values="index_value")
    .rename(
        columns={
            "CPS00000": "headline",
            "CPS00014": "core",
            "CPS07221": "fuel",
            "CPS01000": "food_nab",
        }
    )
    .reset_index()
)
df_inflation.columns.name = None

df_inflation = df_inflation.set_index("date").resample("QE").mean().reset_index()

df_inflation["headline_qoq"] = df_inflation["headline"].pct_change(1) * 100
df_inflation["core_qoq"] = df_inflation["core"].pct_change(1) * 100
df_inflation["fuel_qoq"] = df_inflation["fuel"].pct_change(1) * 100
df_inflation["food_nab_qoq"] = df_inflation["food_nab"].pct_change(1) * 100

df_inflation = normalize_quarter_dates(df_inflation)
df_inflation

In [ ]:
query_fao = """
    SELECT
        date,
        index_value AS ffpi
    FROM fao.main.fao_food_price_indices
    WHERE series_id = 'FFPI'
      AND frequency = 'M'
      AND index_type = 'nominal'
    ORDER BY date
"""

df_fao = con.execute(query_fao).df()

df_fao = df_fao[["date", "ffpi"]].set_index("date").resample("QE").mean().reset_index()

df_fao["ffpi_qoq"] = df_fao["ffpi"].pct_change(1) * 100

df_fao = normalize_quarter_dates(df_fao)
df_fao

In [ ]:
query_brent = """
    SELECT
        date,
        price AS brent_price
    FROM commodity_prices.main.commodity_prices
    WHERE commodity_id = 'BRENT'
      AND frequency = 'M'
    ORDER BY date
"""

df_brent = con.execute(query_brent).df()

df_brent = (
    df_brent[["date", "brent_price"]]
    .set_index("date")
    .resample("QE")
    .mean()
    .reset_index()
)

df_brent["brent_qoq"] = df_brent["brent_price"].pct_change(1) * 100

df_brent = normalize_quarter_dates(df_brent)
df_brent

In [ ]:
query_neer = """
    SELECT
        e.date,
        e.neer
    FROM       country_info.main.countries           AS c
    INNER JOIN eer.main.effective_exchange_rates      AS e  ON e.iso2 = c.iso2
    WHERE c.iso2 = 'ZA'
      AND e.frequency = 'M'
    ORDER BY e.date
"""

df_neer = con.execute(query_neer).df()

df_neer = (
    df_neer[["date", "neer"]].set_index("date").resample("QE").mean().reset_index()
)

df_neer["neer_qoq"] = df_neer["neer"].pct_change(1) * 100

df_neer = normalize_quarter_dates(df_neer)
df_neer

In [ ]:
query_usdzar = """
    SELECT
        date,
        rate AS usdzar
    FROM exchange_rates.main.exchange_rates
    WHERE iso2 = 'ZA'
    ORDER BY date
"""

df_usdzar = con.execute(query_usdzar).df()

df_usdzar = (
    df_usdzar[["date", "usdzar"]].set_index("date").resample("QE").mean().reset_index()
)

df_usdzar["zarusd"] = 1 / df_usdzar["usdzar"]
df_usdzar["usdzar_qoq"] = df_usdzar["usdzar"].pct_change(1) * 100
df_usdzar["zarusd_qoq"] = df_usdzar["zarusd"].pct_change(1) * 100

df_usdzar = normalize_quarter_dates(df_usdzar)
df_usdzar

In [ ]:
con.close()

In [ ]:
dfs = [df_inflation, df_fao, df_brent, df_neer, df_usdzar]

df_merged = (
    reduce(lambda left, right: pd.merge(left, right, on="date", how="outer"), dfs)
    .sort_values("date")
    .reset_index(drop=True)
)

df_merged

In [ ]:
dfs = [
    df_inflation[["date", "headline_qoq", "core_qoq", "fuel_qoq", "food_nab_qoq"]],
    df_fao[["date", "ffpi_qoq"]],
    df_brent[["date", "brent_qoq"]],
    df_neer[["date", "neer_qoq"]],
    df_usdzar[["date", "usdzar_qoq", "zarusd_qoq"]],
]

df_merged = (
    reduce(lambda left, right: pd.merge(left, right, on="date", how="outer"), dfs)
    .sort_values("date")
    .reset_index(drop=True)
)

df_merged

In [ ]:
print(f"Shape: {df_merged.shape}")
print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")
print(f"\nNull counts:\n{df_merged.isnull().sum()}")

In [ ]:
# Clip the data to effective sample window (start and end)

sample_start = "2008-06-30"
sample_end = "2026-03-31"  # cut off at end of Q1 2026; drops any newer Q2/Q3 data that re-running against the live DuckDBs would pull in

df = (
    df_merged.loc[
        (df_merged["date"] >= sample_start) & (df_merged["date"] <= sample_end)
    ]
    .reset_index(drop=True)
    .copy()
)

print(f"Sample: {df['date'].iloc[0].date()} - {df['date'].iloc[-1].date()}")
print(f"Observations: {len(df)}")
print(f"\nNull counts:\n{df.isnull().sum()}")

In [ ]:
series_to_test = [
    "headline_qoq",
    "fuel_qoq",
    "food_nab_qoq",
    "ffpi_qoq",
    "brent_qoq",
    "neer_qoq",
    "usdzar_qoq",
    "zarusd_qoq",
]

results = []

for s in series_to_test:
    x = df[s].dropna()

    # ADF — H0: unit root (non-stationary)
    adf_stat, adf_p, _, _, adf_cv, _ = adfuller(x, autolag="AIC", regression="c")
    adf_reject = adf_p < 0.05

    # KPSS — H0: stationary
    kpss_stat, kpss_p, _, kpss_cv = kpss(x, regression="c", nlags="auto")
    kpss_reject = kpss_p < 0.05

    # Interpret jointly
    if adf_reject and not kpss_reject:
        conclusion = "Stationary I(0)"
    elif not adf_reject and kpss_reject:
        conclusion = "Non-stationary I(1)"
    elif not adf_reject and not kpss_reject:
        conclusion = "Inconclusive (weak)"
    else:
        conclusion = "Inconclusive (conflict)"

    results.append(
        {
            "Series": s,
            "ADF test stat": round(adf_stat, 3),
            "ADF p-value": round(adf_p, 3),
            "ADF reject H0": adf_reject,
            "KPSS test stat": round(kpss_stat, 3),
            "KPSS p-value": round(kpss_p, 3),
            "KPSS reject H0": kpss_reject,
            "Conclusion": conclusion,
        }
    )

df_stationarity = pd.DataFrame(results)
df_stationarity

In [ ]:
# Define variables
endog = df["headline_qoq"].dropna()
exog = df[["neer_qoq", "brent_qoq"]].loc[endog.index]

# Select ARDL lag order
sel = ardl_select_order(
    endog,
    maxlag=4,
    exog=exog,
    maxorder={"neer_qoq": 3, "brent_qoq": 3},
    ic="aic",
    trend="c",
)

# Index = AIC value (float), iloc value = (ar_lags, dl_lags) tuple
best_aic_val = float(sel.aic.index[0])
best_spec = sel.aic.iloc[0]
best_ar = best_spec[0]
best_dl = best_spec[1]

print("=== Empirical Lag selection (AIC) for Headline Inflation Model ===")
print(f"  AR lags on inflation : p = {best_ar}")
print(f"  Lags on NEER         : q = {best_dl.get('neer_qoq')}")
print(f"  Lags on Brent        : r = {best_dl.get('brent_qoq')}")
print(f"  AIC                  : {best_aic_val:.4f}")

print("\n=== Top 5 choices by AIC ===")
for aic_val, spec in sel.aic.iloc[:5].items():
    print(
        f"  p={spec[0]}  q_neer={spec[1].get('neer_qoq')}  r_brent={spec[1].get('brent_qoq')}  AIC={float(aic_val):.4f}"
    )

In [ ]:
# Degrees of freedom for specified model
p, q_neer, r_brent = 4, 2, 1
n_params = 1 + p + (q_neer + 1) + (r_brent + 1)
obs_lost = p
effective_n = len(df) - obs_lost
dof = effective_n - n_params
ratio = effective_n / n_params

print(f"Total observations       : {len(df)}")
print(f"Observations lost to lags: {obs_lost}")
print(f"Effective sample         : {effective_n}")
print(f"Parameters estimated     : {n_params}")
print(f"Degrees of freedom       : {dof}")
print(f"Obs-to-parameter ratio   : {ratio:.1f}")
print(f"\nRule of thumb (>10):     {'Pass' if ratio >= 10 else 'Tight'}")

In [ ]:
# Define variables
endog_food = df["food_nab_qoq"].dropna()
exog_food = df[["ffpi_qoq", "usdzar_qoq"]].loc[endog_food.index]

# AIC grid search
sel_food = ardl_select_order(
    endog_food,
    maxlag=4,
    exog=exog_food,
    maxorder={"ffpi_qoq": 3, "usdzar_qoq": 2},
    ic="aic",
    trend="c",
)

# Index = AIC value (float), iloc value = (ar_lags, dl_lags) tuple
best_aic_val_food = float(sel_food.aic.index[0])
best_spec_food = sel_food.aic.iloc[0]
best_ar_food = best_spec_food[0]
best_dl_food = best_spec_food[1]

print("=== Empirical Lag selection (AIC) for Food & NAB Inflation Model ===")
print(f"  AR lags on food & NAB inflation : p = {best_ar_food}")
print(f"  Lags on FAO FFPI                : q_fao = {best_dl_food.get('ffpi_qoq')}")
print(
    f"  Lags on USDZAR                  : q_usdzar = {best_dl_food.get('usdzar_qoq')}"
)
print(f"  AIC                             : {best_aic_val_food:.4f}")


print("\n=== Top 5 choices by AIC — Food Equation ===")
for aic_val, spec in sel_food.aic.iloc[:10].items():
    print(
        f"  p={spec[0]}  q_fao={spec[1].get('ffpi_qoq')}  "
        f"s_usdzar={spec[1].get('usdzar_qoq')}  AIC={float(aic_val):.4f}"
    )

In [ ]:
# Headline inflation ARDL model, ARDL(4,2,1)
model_emp = ARDL(
    endog, lags=4, exog=exog, order={"neer_qoq": 2, "brent_qoq": 1}, trend="c"
)

# Fit and extract results
res_emp = model_emp.fit(cov_type="HAC", cov_kwds={"maxlags": 4})

print("=== ARDL(4,2,1) Model Results for Headline Inflation ===\n")
print(res_emp.summary())

In [ ]:
# Pass-through scalars
def compute_passthrough(res, ar_lags, neer_lags, brent_lags):
    params = res.params

    # AR persistence sum
    ar_sum = sum(params[f"headline_qoq.L{i}"] for i in ar_lags)
    denom = 1 - ar_sum

    # NEER pass-through (negative sign per GS formula)
    neer_sum = sum(params[f"neer_qoq.L{i}"] for i in neer_lags)
    erpt = -neer_sum / denom

    # Oil pass-through
    brent_sum = sum(params[f"brent_qoq.L{i}"] for i in brent_lags)
    opt = brent_sum / denom

    return ar_sum, denom, erpt, opt


print("=== Empirical ARDL(4, 2, 1) ===")
ar_sum, denom, erpt, opt = compute_passthrough(
    res_emp, ar_lags=[1, 2, 3, 4], neer_lags=[0, 1, 2], brent_lags=[0, 1]
)

print(f"  AR persistence sum (Σβ)      : {ar_sum:.4f}")
print(f"  Persistence multiplier (1-Σβ): {denom:.4f}")
print(f"  ERPT (long-run, per 1% depr) : {erpt:.4f}")
print(f"  OPT  (long-run, per 1% rise) : {opt:.4f}")

In [ ]:
# Residual diagnostics


def build_design_matrix(df, ar_lags, neer_lags, brent_lags):
    cols = {}
    for l in ar_lags:
        cols[f"headline_lag{l}"] = df["headline_qoq"].shift(l)
    for l in neer_lags:
        cols[f"neer_lag{l}"] = df["neer_qoq"].shift(l)
    for l in brent_lags:
        cols[f"brent_lag{l}"] = df["brent_qoq"].shift(l)
    X = pd.DataFrame(cols, index=df.index)
    X = sm.add_constant(X)
    y = df["headline_qoq"]
    combined = pd.concat([y, X], axis=1).dropna()
    return combined.iloc[:, 1:], combined.iloc[:, 0]


def run_diagnostics(res, label, ar_lags, neer_lags, brent_lags):
    resid = res.resid
    X_full, y_full = build_design_matrix(df, ar_lags, neer_lags, brent_lags)
    ols_res = sm.OLS(y_full, X_full).fit()

    print(f"\n{'=' * 60}")
    print(f"DIAGNOSTICS — {label}")
    print(f"{'=' * 60}")

    # ── Ljung-Box ─────────────────────────────────────────────
    lb = acorr_ljungbox(resid, lags=[4, 8], return_df=True)
    print("\n--- Ljung-Box ---")
    for lag in [4, 8]:
        row = lb.loc[lag]
        pval = row["lb_pvalue"]
        print(
            f"  Lag={lag}  stat={row['lb_stat']:.3f}  p={pval:.3f}  "
            f"{'No autocorrelation' if pval >= 0.05 else 'Autocorrelation detected'}"
        )

    # ── Breusch-Godfrey ───────────────────────────────────────
    print("\n--- Breusch-Godfrey ---")
    for lags in [4, 8]:
        bg_stat, bg_p, _, _ = acorr_breusch_godfrey(ols_res, nlags=lags)
        print(
            f"  Lag={lags}  stat={bg_stat:.3f}  p={bg_p:.3f}  "
            f"{'No serial correlation' if bg_p >= 0.05 else 'Serial correlation detected'}"
        )

    # ── Breusch-Pagan ─────────────────────────────────────────
    bp_stat, bp_p, _, _ = het_breuschpagan(resid, X_full)
    print(f"\n--- Breusch-Pagan ---")
    print(
        f"  stat={bp_stat:.3f}  p={bp_p:.3f}  "
        f"{'Homoscedastic' if bp_p >= 0.05 else 'Heteroscedasticity detected'}"
    )

    # ── White's Test ──────────────────────────────────────────
    w_stat, w_p, _, _ = het_white(resid, X_full)
    print(f"\n--- White's Test ---")
    print(
        f"  stat={w_stat:.3f}  p={w_p:.3f}  "
        f"{'Homoscedastic' if w_p >= 0.05 else 'Heteroscedasticity detected'}"
    )

    # ── Jarque-Bera ───────────────────────────────────────────
    jb_stat, jb_p, skew, kurt = jarque_bera(resid)
    print(f"\n--- Jarque-Bera ---")
    print(
        f"  stat={jb_stat:.3f}  p={jb_p:.3f}  skew={skew:.3f}  kurtosis={kurt:.3f}  "
        f"{'Normal residuals' if jb_p >= 0.05 else 'Non-normal residuals'}"
    )

    # ── CUSUM ─────────────────────────────────────────────────
    # Returns: (test_stat, p_value, [(sig_level, critical_value), ...])
    cusum_stat, cusum_p, cusum_cv = breaks_cusumolsresid(resid)
    cv_5pct = dict(cusum_cv)[5]
    print(f"\n--- CUSUM (OLS residuals) ---")
    print(
        f"  stat={cusum_stat:.3f}  p={cusum_p:.3f}  "
        f"5% critical value={cv_5pct:.3f}  "
        f"{'Stable' if cusum_p >= 0.05 else 'Structural instability detected'}"
    )

    return resid


resid_emp = run_diagnostics(
    res_emp,
    "EMPIRICAL ARDL(4, 2, 1)",
    ar_lags=[1, 2, 3, 4],
    neer_lags=[0, 1, 2],
    brent_lags=[0, 1],
)

In [ ]:
# Food and NAB inflation ARDL model, ARDL(4,1,1)
model_food_p4 = ARDL(
    endog_food,
    lags=4,
    exog=exog_food,
    order={"ffpi_qoq": 1, "usdzar_qoq": 1},
    trend="c",
).fit(cov_type="HAC", cov_kwds={"maxlags": 4})

print("=" * 60)
print("FOOD EQUATION — ARDL(AR=4, FAO=1, USDZAR=1)")
print("=" * 60)
print(model_food_p4.summary())

In [ ]:
def compute_fpt(res, ar_lags, fao_lags):
    params = res.params

    ar_sum = sum(params[f"food_nab_qoq.L{i}"] for i in ar_lags)
    denom = 1 - ar_sum

    fao_sum = sum(params[f"ffpi_qoq.L{j}"] for j in fao_lags)
    fpt = fao_sum / denom

    return ar_sum, denom, fao_sum, fpt


ar_sum_food, denom_food, fao_sum_food, fpt = compute_fpt(
    model_food_p4, ar_lags=[1, 2, 3, 4], fao_lags=[0, 1]
)

print("=== Food Pass-Through Scalar===")
print(f"  AR persistence sum (Σβ)              : {ar_sum_food:.4f}")
print(f"  Persistence multiplier (1−Σβ)        : {denom_food:.4f}")
print(f"  FAO coefficient sum (Σγ: L0+L1)      : {fao_sum_food:.4f}")
print(f"  FPT (food sub-index level)            : {fpt:.4f}")
print(f"\n  Note: FPT at food sub-index level.")
print(f"  Re-scale to headline CPI: FPT × food & NAB basket weight")

In [ ]:
FOOD_NAB_WEIGHT = 0.1823

fpt_headline = fpt * FOOD_NAB_WEIGHT

print("=== Basket Weights ===")
print(
    f"  Food & NAB weight in headline CPI : {FOOD_NAB_WEIGHT:.4f} ({FOOD_NAB_WEIGHT * 100:.2f}%)"
)

print("\n=== FPT Re-scaled to Headline CPI ===")
print(f"  FPT at food sub-index level       : {fpt:.4f}")
print(f"  Food & NAB basket weight          : {FOOD_NAB_WEIGHT:.4f}")
print(f"  FPT at headline CPI level         : {fpt_headline:.4f}")
print(f"\n  Interpretation: a sustained 1% increase in the FAO Food Price Index")
print(f"  raises ZA headline CPI by {fpt_headline:.4f}% in the long run.")

In [ ]:
# Residual diagnostics


def build_design_matrix_food(df, ar_lags, fao_lags, usd_lags):
    cols = {}
    for l in ar_lags:
        cols[f"food_lag{l}"] = df["food_nab_qoq"].shift(l)
    for l in fao_lags:
        cols[f"fao_lag{l}"] = df["ffpi_qoq"].shift(l)
    for l in usd_lags:
        cols[f"usd_lag{l}"] = df["usdzar_qoq"].shift(l)
    X = pd.DataFrame(cols, index=df.index)
    X = sm.add_constant(X)
    y = df["food_nab_qoq"]
    combined = pd.concat([y, X], axis=1).dropna()
    return combined.iloc[:, 1:], combined.iloc[:, 0]


def run_diagnostics_food(res, label, ar_lags, fao_lags, usd_lags):
    resid = res.resid
    X_full, y_full = build_design_matrix_food(df, ar_lags, fao_lags, usd_lags)
    ols_res = sm.OLS(y_full, X_full).fit()

    print(f"\n{'=' * 60}")
    print(f"DIAGNOSTICS — {label}")
    print(f"{'=' * 60}")

    lb = acorr_ljungbox(resid, lags=[4, 8], return_df=True)
    print("\n--- Ljung-Box ---")
    for lag in [4, 8]:
        row = lb.loc[lag]
        pval = row["lb_pvalue"]
        print(
            f"  Lag={lag}  stat={row['lb_stat']:.3f}  p={pval:.3f}  "
            f"{'No autocorrelation' if pval >= 0.05 else 'Autocorrelation detected'}"
        )

    print("\n--- Breusch-Godfrey ---")
    for lags in [4, 8]:
        bg_stat, bg_p, _, _ = acorr_breusch_godfrey(ols_res, nlags=lags)
        print(
            f"  Lag={lags}  stat={bg_stat:.3f}  p={bg_p:.3f}  "
            f"{'No serial correlation' if bg_p >= 0.05 else 'Serial correlation detected'}"
        )

    bp_stat, bp_p, _, _ = het_breuschpagan(resid, X_full)
    print(f"\n--- Breusch-Pagan ---")
    print(
        f"  stat={bp_stat:.3f}  p={bp_p:.3f}  "
        f"{'Homoscedastic' if bp_p >= 0.05 else 'Heteroscedasticity detected'}"
    )

    w_stat, w_p, _, _ = het_white(resid, X_full)
    print(f"\n--- White's Test ---")
    print(
        f"  stat={w_stat:.3f}  p={w_p:.3f}  "
        f"{'Homoscedastic' if w_p >= 0.05 else 'Heteroscedasticity detected'}"
    )

    jb_stat, jb_p, skew, kurt = jarque_bera(resid)
    print(f"\n--- Jarque-Bera ---")
    print(
        f"  stat={jb_stat:.3f}  p={jb_p:.3f}  skew={skew:.3f}  kurtosis={kurt:.3f}  "
        f"{'Normal residuals' if jb_p >= 0.05 else 'Non-normal residuals'}"
    )

    cusum_stat, cusum_p, cusum_cv = breaks_cusumolsresid(resid)
    cv_5pct = dict(cusum_cv)[5]
    print(f"\n--- CUSUM (OLS residuals) ---")
    print(
        f"  stat={cusum_stat:.3f}  p={cusum_p:.3f}  "
        f"5% critical value={cv_5pct:.3f}  "
        f"{'Stable' if cusum_p >= 0.05 else 'Structural instability detected'}"
    )

    return resid


ar_lags_food = [1, 2, 3, 4]
fao_lags_food = [0, 1]
usd_lags_food = [0, 1]

resid_food = run_diagnostics_food(
    model_food_p4,
    label="FOOD ARDL(AR=4, FAO=1, USDZAR=1)",
    ar_lags=ar_lags_food,
    fao_lags=fao_lags_food,
    usd_lags=usd_lags_food,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Food Equation Residuals — ARDL(AR=4, FAO=1, USDZAR=1)", fontsize=12)

axes[0].plot(resid_food.values)
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_title("Residuals over time")
axes[0].set_ylabel("Residual")

axes[1].hist(resid_food.values, bins=20, edgecolor="black")
axes[1].set_title("Residual histogram")
axes[1].set_xlabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
# Prepare data matrices for rolling regressions

if "date" in df.columns:
    df = df.set_index("date")
df.index = pd.to_datetime(df.index)

# Eq.1 — headline CPI: ARDL(4, 2, 1)
X_eq1 = pd.DataFrame(
    {
        "const": 1.0,
        "headline_L1": df["headline_qoq"].shift(1),
        "headline_L2": df["headline_qoq"].shift(2),
        "headline_L3": df["headline_qoq"].shift(3),
        "headline_L4": df["headline_qoq"].shift(4),
        "neer_L0": df["neer_qoq"],
        "neer_L1": df["neer_qoq"].shift(1),
        "neer_L2": df["neer_qoq"].shift(2),
        "brent_L0": df["brent_qoq"],
        "brent_L1": df["brent_qoq"].shift(1),
    },
    index=df.index,
).dropna()

y_eq1 = df["headline_qoq"].loc[X_eq1.index]

# Eq.5 — food & NAB CPI: ARDL(4, 1, 1)
X_eq5 = pd.DataFrame(
    {
        "const": 1.0,
        "food_L1": df["food_nab_qoq"].shift(1),
        "food_L2": df["food_nab_qoq"].shift(2),
        "food_L3": df["food_nab_qoq"].shift(3),
        "food_L4": df["food_nab_qoq"].shift(4),
        "fao_L0": df["ffpi_qoq"],
        "fao_L1": df["ffpi_qoq"].shift(1),
        "usdzar_L0": df["usdzar_qoq"],
        "usdzar_L1": df["usdzar_qoq"].shift(1),
    },
    index=df.index,
).dropna()

y_eq5 = df["food_nab_qoq"].loc[X_eq5.index]

# Verify
print(
    f"Eq.1: X {X_eq1.shape}, y {y_eq1.shape}, "
    f"sample {X_eq1.index[0].date()} - {X_eq1.index[-1].date()}"
)
print(
    f"Eq.5: X {X_eq5.shape}, y {y_eq5.shape}, "
    f"sample {X_eq5.index[0].date()} - {X_eq5.index[-1].date()}"
)

In [ ]:
# Set window for observations
WINDOW = 40

# Headline CPI rolling regression ARDL(4, 2, 1)
rols_eq1 = RollingOLS(y_eq1, X_eq1, window=WINDOW, min_nobs=WINDOW).fit()

# Extract parameters and R² values
params_eq1 = rols_eq1.params.copy()
r2_eq1 = rols_eq1.rsquared.copy()
r2_adj_eq1 = rols_eq1.rsquared_adj.copy()

# Food & NAB CPI rolling regression ARDL(4, 1, 1)
rols_eq5 = RollingOLS(y_eq5, X_eq5, window=WINDOW, min_nobs=WINDOW).fit()

# Extract parameters and R² values
params_eq5 = rols_eq5.params.copy()
r2_eq5 = rols_eq5.rsquared.copy()
r2_adj_eq5 = rols_eq5.rsquared_adj.copy()

# Verify
for label, p, r2, r2a in [
    ("Eq.1", params_eq1, r2_eq1, r2_adj_eq1),
    ("Eq.5", params_eq5, r2_eq5, r2_adj_eq5),
]:
    valid = p.dropna()
    print(f"\n{label}")
    print(
        f"  params:       {valid.shape} — "
        f"{valid.index[0].date()} → {valid.index[-1].date()}"
    )
    print(f"  R² range:     {r2.dropna().min():.3f} – {r2.dropna().max():.3f}")
    print(f"  Adj R² range: {r2a.dropna().min():.3f} – {r2a.dropna().max():.3f}")

In [ ]:
# Check headline CPI ARDL(4, 2, 1) rolling regression results
params_eq1.tail(30)

In [ ]:
# Check food & NAB CPI ARDL(4, 1, 1) rolling regression results
params_eq5.tail(30)

In [ ]:
# Check R-squared results
# r2_eq1
# r2_adj_eq1
# r2_eq5
# r2_adj_eq5

In [ ]:
# Confirm the headline inflation ARDL(4, 2, 1) parameters
res_emp.params

In [ ]:
# Confirm the food and NAB inflation ARDL(4, 1, 1) parameters
model_food_p4.params

In [ ]:
# Burn-in backfill

# Headline inflation mapping
eq1_map = {
    "const": "const",
    "headline_qoq.L1": "headline_L1",
    "headline_qoq.L2": "headline_L2",
    "headline_qoq.L3": "headline_L3",
    "headline_qoq.L4": "headline_L4",
    "neer_qoq.L0": "neer_L0",
    "neer_qoq.L1": "neer_L1",
    "neer_qoq.L2": "neer_L2",
    "brent_qoq.L0": "brent_L0",
    "brent_qoq.L1": "brent_L1",
}

# Food & NAB inflation mapping
eq5_map = {
    "const": "const",
    "food_nab_qoq.L1": "food_L1",
    "food_nab_qoq.L2": "food_L2",
    "food_nab_qoq.L3": "food_L3",
    "food_nab_qoq.L4": "food_L4",
    "ffpi_qoq.L0": "fao_L0",
    "ffpi_qoq.L1": "fao_L1",
    "usdzar_qoq.L0": "usdzar_L0",
    "usdzar_qoq.L1": "usdzar_L1",
}

# Identify burn-in rows
burnin_eq1 = params_eq1[params_eq1.isna().any(axis=1)].index
burnin_eq5 = params_eq5[params_eq5.isna().any(axis=1)].index

# Backfill headline inflation parameters
for ardl_name, roll_name in eq1_map.items():
    params_eq1.loc[burnin_eq1, roll_name] = res_emp.params[ardl_name]

# Backfill food & NAB inflation parameters
for ardl_name, roll_name in eq5_map.items():
    params_eq5.loc[burnin_eq5, roll_name] = model_food_p4.params[ardl_name]

# Verify backfill — no NaNs should remain
print(f"Eq.1 NaNs remaining: {params_eq1.isna().sum().sum()}")
print(f"Eq.5 NaNs remaining: {params_eq5.isna().sum().sum()}")
print(f"\nEq.1 burn-in rows filled: {len(burnin_eq1)}")
print(f"Eq.5 burn-in rows filled: {len(burnin_eq5)}")

In [ ]:
# Check backfilled headline inflation
params_eq1

In [ ]:
# Check backfilled food & NAB inflation
params_eq5

In [ ]:
# Headline inflation — AR persistence sum
ar_sum_eq1 = (
    params_eq1["headline_L1"]
    + params_eq1["headline_L2"]
    + params_eq1["headline_L3"]
    + params_eq1["headline_L4"]
)

# ERPT(t)
erpt = -(params_eq1["neer_L0"] + params_eq1["neer_L1"] + params_eq1["neer_L2"]) / (
    1 - ar_sum_eq1
)

# OPT(t)
opt = (params_eq1["brent_L0"] + params_eq1["brent_L1"]) / (1 - ar_sum_eq1)

# Food and NABAR persistence sum
ar_sum_eq5 = (
    params_eq5["food_L1"]
    + params_eq5["food_L2"]
    + params_eq5["food_L3"]
    + params_eq5["food_L4"]
)

# FPT(t)
fpt = (params_eq5["fao_L0"] + params_eq5["fao_L1"]) / (1 - ar_sum_eq5)

fpt_headline = fpt * FOOD_NAB_WEIGHT

# Compile results into a DataFrame
scalars = pd.DataFrame(
    {
        "erpt": erpt,
        "opt": opt,
        "fpt_food": fpt,
        "fpt_headline": fpt_headline,
    }
)

scalars

In [ ]:
IRF_HORIZON = 12


def build_irf(ar_coefs, impact_coef, horizon=IRF_HORIZON):
    """
    Build IRF to a 1% shock in a variable.

    ar_coefs:    array of AR coefficients [β1, β2, β3, β4]
    impact_coef: contemporaneous coefficient on the shocked variable (γ0)
    horizon:     number of quarters ahead
    """
    irf = np.zeros(horizon)
    irf[0] = impact_coef
    for h in range(1, horizon):
        for i, beta in enumerate(ar_coefs):
            lag = h - (i + 1)
            if lag >= 0:
                irf[h] += beta * irf[lag]
    return irf


def compute_contribution(irf, actual_changes, t_idx, horizon=IRF_HORIZON):
    """
    Convolve IRF with actual historical changes at quarter t.

    irf:            array of IRF values at horizons 0..11
    actual_changes: full Series of actual quarterly changes
    t_idx:          integer position of current quarter
    """
    effect = 0.0
    for h in range(horizon):
        idx = t_idx - h
        if idx >= 0:
            effect += irf[h] * actual_changes.iloc[idx]
    return effect


# Actual change series — aligned to estimation sample
dates = X_eq1.index
neer_actual = df["neer_qoq"].loc[dates]
brent_actual = df["brent_qoq"].loc[dates]
fao_actual = df["ffpi_qoq"].loc[dates]

# Output series
neer_effect = pd.Series(index=dates, dtype=float)
oil_effect = pd.Series(index=dates, dtype=float)
food_effect = pd.Series(index=dates, dtype=float)

# AR column names
ar_cols_eq1 = ["headline_L1", "headline_L2", "headline_L3", "headline_L4"]
ar_cols_eq5 = ["food_L1", "food_L2", "food_L3", "food_L4"]

for i, t in enumerate(dates):
    # AR coefficients at t
    ar_eq1 = params_eq1.loc[t, ar_cols_eq1].values
    ar_eq5 = params_eq5.loc[t, ar_cols_eq5].values

    # Build IRFs
    irf_neer = build_irf(ar_eq1, params_eq1.loc[t, "neer_L0"])
    irf_brent = build_irf(ar_eq1, params_eq1.loc[t, "brent_L0"])
    irf_fao = build_irf(ar_eq5, params_eq5.loc[t, "fao_L0"])

    # Compute contributions
    neer_effect.iloc[i] = compute_contribution(irf_neer, neer_actual, i)
    oil_effect.iloc[i] = compute_contribution(irf_brent, brent_actual, i)
    food_effect.iloc[i] = compute_contribution(irf_fao, fao_actual, i) * FOOD_NAB_WEIGHT

# Verify
print(
    f"neer_effect:  {neer_effect.isna().sum()} NaNs, "
    f"range {neer_effect.min():.3f} – {neer_effect.max():.3f}"
)
print(
    f"oil_effect:   {oil_effect.isna().sum()} NaNs, "
    f"range {oil_effect.min():.3f} – {oil_effect.max():.3f}"
)
print(
    f"food_effect:  {food_effect.isna().sum()} NaNs, "
    f"range {food_effect.min():.3f} – {food_effect.max():.3f}"
)

In [ ]:
contributions = pd.DataFrame(
    {
        "neer_effect": neer_effect,
        "oil_effect": oil_effect,
        "food_effect": food_effect,
    }
)

contributions

In [ ]:
# Construct DIPI series as headline CPI change minus contributions
dipi = df["headline_qoq"].loc[dates] - neer_effect - oil_effect - food_effect

# Name the object
dipi.name = "dipi"

# Verify construction
print(f"DIPI: {dipi.isna().sum()} NaNs")
print(f"Sample range: {dipi.index[0].date()} - {dipi.index[-1].date()}")
print(f"Range: {dipi.min():.3f} – {dipi.max():.3f}")
print(f"Mean:  {dipi.mean():.3f}")

# Return the dataframe
dipi

In [ ]:
# Annualised contribution estimates - rolling 4 quarter sum
yoy = pd.DataFrame(
    {
        "headline_yoy": df["headline_qoq"].loc[dates].rolling(4).sum(),
        "neer_effect_yoy": neer_effect.rolling(window=4).sum(),
        "oil_effect_yoy": oil_effect.rolling(window=4).sum(),
        "food_effect_yoy": food_effect.rolling(window=4).sum(),
        "dipi_yoy": dipi.rolling(window=4).sum(),
    }
)

# Verify additivity - contributions should sum to headline CPI change at each quarter
yoy["check"] = (
    yoy["neer_effect_yoy"]
    + yoy["oil_effect_yoy"]
    + yoy["food_effect_yoy"]
    + yoy["dipi_yoy"]
)

yoy["residual"] = yoy["headline_yoy"] - yoy["check"]

print(f"NaNs per series:\n{yoy.isna().sum()}")
print(f"\nAdditivity check — max residual: {yoy['residual'].abs().max():.6f}")
print(f"Sample: {yoy.dropna().index[0].date()} - {yoy.dropna().index[-1].date()}")

# Display the annualised contributions
yoy.tail(20)

In [ ]:
# Drop check and residual columns from dataframe
yoy = yoy.drop(columns=["check", "residual"])

In [ ]:
# Hold-constant projections

# Generate future dates at quarterly frequency
proj_dates = pd.date_range(start="2026-06-30", end="2027-12-31", freq="QE")

# Extend actual change series with zeros as changes for projection assumptions across future quarters
neer_ext = pd.concat([neer_actual, pd.Series(0.0, index=proj_dates)])
brent_ext = pd.concat([brent_actual, pd.Series(0.0, index=proj_dates)])
fao_ext = pd.concat([fao_actual, pd.Series(0.0, index=proj_dates)])

# Extract last available rolling coefficient vector for held constant projections
last_ar_eq1 = params_eq1.loc["2026-03-31", ar_cols_eq1].values
last_ar_eq5 = params_eq5.loc["2026-03-31", ar_cols_eq5].values
last_neer_L0 = params_eq1.loc["2026-03-31", "neer_L0"]
last_brent_L0 = params_eq1.loc["2026-03-31", "brent_L0"]
last_fao_L0 = params_eq5.loc["2026-03-31", "fao_L0"]

# Builds IRF's from the last coefficient vector - fixed for all projection quarters
irf_neer_proj = build_irf(last_ar_eq1, last_neer_L0)
irf_brent_proj = build_irf(last_ar_eq1, last_brent_L0)
irf_fao_proj = build_irf(last_ar_eq5, last_fao_L0)

# Compute contributions for projection quarters
neer_effect_proj = pd.Series(index=proj_dates, dtype=float)
oil_effect_proj = pd.Series(index=proj_dates, dtype=float)
food_effect_proj = pd.Series(index=proj_dates, dtype=float)

all_dates = pd.Index(dates.tolist() + proj_dates.tolist())

for i, t in enumerate(proj_dates):
    t_idx = len(dates) + i

    neer_effect_proj.iloc[i] = compute_contribution(irf_neer_proj, neer_ext, t_idx)
    oil_effect_proj.iloc[i] = compute_contribution(irf_brent_proj, brent_ext, t_idx)
    food_effect_proj.iloc[i] = (
        compute_contribution(irf_fao_proj, fao_ext, t_idx) * FOOD_NAB_WEIGHT
    )

# Extend full contribution series
# Not creating new historical contributions
# Adding the projected values to the end of the historical contributions
# pd.concat is joining the two series end-on-end, aligning by index (dates)
neer_effect_full = pd.concat([neer_effect, neer_effect_proj])
oil_effect_full = pd.concat([oil_effect, oil_effect_proj])
food_effect_full = pd.concat([food_effect, food_effect_proj])

# Extend headline CPI with NaNs for projection quarters - not observed
headline_full = pd.concat(
    [df["headline_qoq"].loc[dates], pd.Series(np.nan, index=proj_dates)]
)

# Extend DIPI - projection quarters have no observed headline CPI
# DIPI for projections = 0 (no domestic assumption) + external contributions
# Flagged separately - we store contributions only for projection period
dipi_full = pd.concat([dipi, pd.Series(np.nan, index=proj_dates)])

# Flag projected quarters
is_projected = pd.Series(False, index=dates).reindex(
    dates.tolist() + proj_dates.tolist(), fill_value=False
)
is_projected.loc[proj_dates] = True

# Annualise extended series
yoy_full = pd.DataFrame(
    {
        "headline_yoy": headline_full.rolling(4).sum(),
        "neer_effect_yoy": neer_effect_full.rolling(4).sum(),
        "oil_effect_yoy": oil_effect_full.rolling(4).sum(),
        "food_effect_yoy": food_effect_full.rolling(4).sum(),
        "dipi_yoy": dipi_full.rolling(4).sum(),
        "is_projected": is_projected,
    }
)

yoy_full.tail(20)

In [ ]:
# Verify the new series
print(f"Extended series: {len(yoy_full)} rows")
print(
    f"Sample: {yoy_full.dropna(subset=['neer_effect_yoy']).index[0].date()} "
    f"→ {yoy_full.index[-1].date()}"
)
print(f"Projection quarters: {is_projected.sum()}")
print(f"\nProjected contributions (2026 Q2 → 2027 Q4):")
print(
    yoy_full[yoy_full["is_projected"]][
        ["neer_effect_yoy", "oil_effect_yoy", "food_effect_yoy"]
    ].round(3)
)

In [ ]:
# Oil price shocks

# Each scenario defines the quarterly brent change for the projection period
# The first two projection quarters are shocked, with the remainder held at zero change

# Combine historical dates and projection dates into a single index for the full dataset
# 68 historical + 7 projection quarters = 75 total rows
dates_full = pd.Index(dates.tolist() + proj_dates.tolist())
n_hist = len(dates)

# Define shock scenarios for brent price changes in projection quarters
scenarios = {
    "Baseline": [0.0, 0.0],
    "Mild (+5%)": [5.0, 5.0],
    "Moderate (+10%)": [10.0, 10.0],
    "Severe (+20%)": [20.0, 20.0],
}

# Build the IRF for brent price shocks based on last available coefficients
last_t = dates[-1]
ar_eq1_last = params_eq1.loc[last_t, ar_cols_eq1].values

irf_brent_last = build_irf(ar_eq1_last, params_eq1.loc[last_t, "brent_L0"])

scenario_results = {}

for name, shock_path in scenarios.items():
    # Build projection Brent series — shock for first 2 quarters, zero after
    brent_scenario = pd.Series(0.0, index=proj_dates)
    for i, val in enumerate(shock_path):
        brent_scenario.iloc[i] = val

    # Extend actual Brent with scenario projection
    brent_full_scen = pd.concat([brent_actual, brent_scenario])

    # Historical period — reuse the time-varying rolling-coefficient contributions
    # (identical across scenarios). Recomputing history with the fixed 2026 Q1 IRF
    # made the scenario "Baseline" diverge from the hold-constant path in any yoy
    # quarter whose 4-quarter window includes pre-2026 Q1 history, and made the
    # historical bars inconsistent with the headline decomposition.

    # Projection period — last IRF, scenario Brent path
    oil_proj = pd.Series(index=proj_dates, dtype=float)
    for i, t in enumerate(proj_dates):
        idx = n_hist + i
        oil_proj.iloc[i] = compute_contribution(irf_brent_last, brent_full_scen, idx)

    oil_effect_scen = pd.concat([oil_effect, oil_proj])

    # Annualise
    scenario_results[name] = oil_effect_scen.rolling(4).sum()

scenario_df = pd.DataFrame(scenario_results, index=dates_full)

# The zero-shock scenario must be identical to the hold-constant oil path
baseline_diff = (scenario_df["Baseline"] - yoy_full["oil_effect_yoy"]).abs().max()
print(f"Baseline vs hold-constant oil path — max abs diff: {baseline_diff:.2e}")
assert baseline_diff < 1e-9, (
    "Scenario Baseline diverges from the hold-constant oil path"
)

# Trim to projection period + history for context
scenario_df = scenario_df.loc["2024-03-31":]

print("Scenario oil contributions - annualised (yoy) — projection period + history:")
scenario_df

In [ ]:
# Linked shock scenarios — Oil, FAO and NEER

# Elasticities derived from full-sample level regressions
FAO_ELASTICITY = 0.45  # 1% Brent → +0.45% FAO
NEER_ELASTICITY = -0.54  # 1% Brent → -0.54% NEER

# Shock scenarios defined by Brent quarterly change — FAO and NEER scaled automatically
linked_scenarios = {
    "Baseline": [0.0, 0.0],
    "Mild (+5%)": [5.0, 5.0],
    "Moderate (+10%)": [10.0, 10.0],
    "Severe (+20%)": [20.0, 20.0],
}

# Build IRFs from last available coefficient vector — fixed for all scenarios
ar_eq5_last = params_eq5.loc[last_t, ar_cols_eq5].values

irf_neer_last = build_irf(ar_eq1_last, params_eq1.loc[last_t, "neer_L0"])

irf_fao_last = build_irf(ar_eq5_last, params_eq5.loc[last_t, "fao_L0"])

# ── Run each linked scenario ──────────────────────────────────────────────────

linked_results = {}

for name, shock_path in linked_scenarios.items():
    # Scale FAO and NEER shocks from Brent shock via elasticities
    fao_path = [s * FAO_ELASTICITY for s in shock_path]
    neer_path = [s * NEER_ELASTICITY for s in shock_path]

    # Build projection series for each variable — shock first 2 quarters, zero after
    brent_scenario = pd.Series(0.0, index=proj_dates)
    fao_scenario = pd.Series(0.0, index=proj_dates)
    neer_scenario = pd.Series(0.0, index=proj_dates)

    for i, (b, f, n) in enumerate(zip(shock_path, fao_path, neer_path)):
        brent_scenario.iloc[i] = b
        fao_scenario.iloc[i] = f
        neer_scenario.iloc[i] = n

    # Extend actual series with scenario projections
    brent_full_scen = pd.concat([brent_actual, brent_scenario])
    fao_full_scen = pd.concat([fao_actual, fao_scenario])
    neer_full_scen = pd.concat([neer_actual, neer_scenario])

    # Initialise contribution series
    oil_effect_scen = pd.Series(index=dates_full, dtype=float)
    food_effect_scen = pd.Series(index=dates_full, dtype=float)
    neer_effect_scen = pd.Series(index=dates_full, dtype=float)

    # Historical period
    for i, t in enumerate(dates):
        oil_effect_scen.iloc[i] = compute_contribution(
            irf_brent_last, brent_full_scen, i
        )
        food_effect_scen.iloc[i] = (
            compute_contribution(irf_fao_last, fao_full_scen, i) * FOOD_NAB_WEIGHT
        )
        neer_effect_scen.iloc[i] = compute_contribution(
            irf_neer_last, neer_full_scen, i
        )

    # Projection period
    for i, t in enumerate(proj_dates):
        idx = n_hist + i
        oil_effect_scen.iloc[idx] = compute_contribution(
            irf_brent_last, brent_full_scen, idx
        )
        food_effect_scen.iloc[idx] = (
            compute_contribution(irf_fao_last, fao_full_scen, idx) * FOOD_NAB_WEIGHT
        )
        neer_effect_scen.iloc[idx] = compute_contribution(
            irf_neer_last, neer_full_scen, idx
        )

    # Annualise
    oil_yoy = oil_effect_scen.rolling(4).sum()
    food_yoy = food_effect_scen.rolling(4).sum()
    neer_yoy = neer_effect_scen.rolling(4).sum()
    total_yoy = oil_yoy + food_yoy + neer_yoy

    linked_results[name] = {
        "oil": oil_yoy,
        "food": food_yoy,
        "neer": neer_yoy,
        "total": total_yoy,
    }

# Compile into DataFrames — one per channel plus total

linked_oil_df = pd.DataFrame(
    {k: v["oil"] for k, v in linked_results.items()}, index=dates_full
)
linked_food_df = pd.DataFrame(
    {k: v["food"] for k, v in linked_results.items()}, index=dates_full
)
linked_neer_df = pd.DataFrame(
    {k: v["neer"] for k, v in linked_results.items()}, index=dates_full
)
linked_total_df = pd.DataFrame(
    {k: v["total"] for k, v in linked_results.items()}, index=dates_full
)

# Trim to projection period + 4 quarters history for context
trim_date = "2024-03-31"
linked_oil_df = linked_oil_df.loc[trim_date:]
linked_food_df = linked_food_df.loc[trim_date:]
linked_neer_df = linked_neer_df.loc[trim_date:]
linked_total_df = linked_total_df.loc[trim_date:]

print("Linked scenario — total external contribution (%yoy):")
print(linked_total_df.round(3))

In [ ]:
# Per-channel contributions by scenario
for scenario_name in linked_scenarios.keys():
    print(f"\n{scenario_name} — channel breakdown (%yoy):")
    channel_df = pd.DataFrame(
        {
            "Oil": linked_results[scenario_name]["oil"],
            "Food": linked_results[scenario_name]["food"],
            "NEER": linked_results[scenario_name]["neer"],
            "Total": linked_results[scenario_name]["total"],
        }
    ).loc[trim_date:]
    print(channel_df.round(3))

In [ ]:
# Estimate AR(1) persistence for each external driver
# Runs on existing df series. No new data construction.

from statsmodels.tsa.ar_model import AutoReg


def estimate_ar1(series, name):
    s = series.dropna()
    res = AutoReg(s, lags=1, old_names=False).fit()
    rho = res.params.iloc[1]
    print(
        f"  {name:<12}  ρ = {rho:.4f}  "
        f"(t = {res.tvalues.iloc[1]:.2f},  "
        f"p = {res.pvalues.iloc[1]:.3f})"
    )
    return rho


print("AR(1) persistence estimates — quarterly log-changes")
print("=" * 55)
rho_brent = estimate_ar1(df["brent_qoq"], "Brent")
rho_fao = estimate_ar1(df["ffpi_qoq"], "FAO")
rho_neer = estimate_ar1(df["neer_qoq"], "NEER")

In [ ]:
# IRF simulation from full-sample ARDL coefficients ─────────
# Uses res_emp (Eq.1) and model_food_p4 (Eq.5) — not rolling window.

N_IRF = 20
horizons = np.arange(N_IRF)

# ── Extract full-sample Eq.1 coefficients ─────────────────────────────────
b_eq1 = res_emp.params

ar_fs = np.array(
    [
        b_eq1["headline_qoq.L1"],
        b_eq1["headline_qoq.L2"],
        b_eq1["headline_qoq.L3"],
        b_eq1["headline_qoq.L4"],
    ]
)
g_oil = np.array([b_eq1["brent_qoq.L0"], b_eq1["brent_qoq.L1"]])
g_neer = np.array([b_eq1["neer_qoq.L0"], b_eq1["neer_qoq.L1"], b_eq1["neer_qoq.L2"]])

# ── Extract full-sample Eq.5 coefficients ─────────────────────────────────
b_eq5 = model_food_p4.params

ar_fs_food = np.array(
    [
        b_eq5["food_nab_qoq.L1"],
        b_eq5["food_nab_qoq.L2"],
        b_eq5["food_nab_qoq.L3"],
        b_eq5["food_nab_qoq.L4"],
    ]
)
g_fao = np.array([b_eq5["ffpi_qoq.L0"], b_eq5["ffpi_qoq.L1"]])


# ── Generic recursive IRF simulator ───────────────────────────────────────
def simulate_irf(ar_coefs, shock_coefs, shock_size, n=N_IRF):
    """
    Simulate ARDL impulse response to a one-time shock.

    ar_coefs    : array of AR coefficients [β₁, β₂, ..., βₚ]
    shock_coefs : array of distributed lag coefficients [γ₀, γ₁, ..., γₛ]
    shock_size  : scalar magnitude of the impulse at t=0
    """
    pi = np.zeros(n)
    for t in range(n):
        # Distributed lag contribution — only fires while lag index in range
        for j, g in enumerate(shock_coefs):
            if t == j:
                pi[t] += g * shock_size
        # AR contribution
        for i, b in enumerate(ar_coefs):
            if t - (i + 1) >= 0:
                pi[t] += b * pi[t - (i + 1)]
    return pi


# ── Compute IRFs for each channel at each scenario shock size ──────────────
# Shock sizes from linked_scenarios — extract non-baseline entries
irf_shock_labels = [k for k in linked_scenarios if k != "Baseline"]
irf_shock_sizes = [linked_scenarios[k][0] for k in irf_shock_labels]

irf_results = {}

for label, s_brent in zip(irf_shock_labels, irf_shock_sizes):
    s_fao = s_brent * FAO_ELASTICITY
    s_neer = s_brent * NEER_ELASTICITY

    irf_results[label] = {
        "oil": simulate_irf(ar_fs, g_oil, s_brent),
        "neer": simulate_irf(ar_fs, g_neer, s_neer),
        "food": simulate_irf(ar_fs_food, g_fao, s_fao) * FOOD_NAB_WEIGHT,
    }


# ── Shock persistence paths (AR(1) decay) ─────────────────────────────────
def ar1_path(shock_size, rho, n=N_IRF):
    return np.array([shock_size * (rho**t) for t in range(n)])


shock_paths = {
    label: {
        "brent": ar1_path(linked_scenarios[label][0], rho_brent),
        "fao": ar1_path(linked_scenarios[label][0] * FAO_ELASTICITY, rho_fao),
        "neer": ar1_path(linked_scenarios[label][0] * NEER_ELASTICITY, rho_neer),
    }
    for label in irf_shock_labels
}

print("IRF simulation complete.")
print(f"  Horizons : {N_IRF} quarters")
print(f"  Scenarios: {irf_shock_labels}")
print(f"\n  Full-sample AR coefficients (Eq.1): {ar_fs.round(4)}")
print(f"  Oil DL coefficients                : {g_oil.round(4)}")
print(f"  NEER DL coefficients               : {g_neer.round(4)}")
print(f"\n  Full-sample AR coefficients (Eq.5): {ar_fs_food.round(4)}")
print(f"  FAO DL coefficients                : {g_fao.round(4)}")

In [ ]:
print("res_emp parameter names:")
print(res_emp.params.index.tolist())

print("\nmodel_food_p4 parameter names:")
print(model_food_p4.params.index.tolist())

In [ ]:
# IRF simulation from full-sample ARDL coefficients
from scipy.interpolate import make_interp_spline

N_IRF = 24
horizons = np.arange(N_IRF)

# ── Extract full-sample Eq.1 coefficients ─────────────────────────────────
b_eq1 = res_emp.params

ar_fs = np.array(
    [
        b_eq1["headline_qoq.L1"],
        b_eq1["headline_qoq.L2"],
        b_eq1["headline_qoq.L3"],
        b_eq1["headline_qoq.L4"],
    ]
)
g_oil = np.array([b_eq1["brent_qoq.L0"], b_eq1["brent_qoq.L1"]])
g_neer = np.array([b_eq1["neer_qoq.L0"], b_eq1["neer_qoq.L1"], b_eq1["neer_qoq.L2"]])

# ── Extract full-sample Eq.5 coefficients ─────────────────────────────────
b_eq5 = model_food_p4.params

ar_fs_food = np.array(
    [
        b_eq5["food_nab_qoq.L1"],
        b_eq5["food_nab_qoq.L2"],
        b_eq5["food_nab_qoq.L3"],
        b_eq5["food_nab_qoq.L4"],
    ]
)
g_fao = np.array([b_eq5["ffpi_qoq.L0"], b_eq5["ffpi_qoq.L1"]])


# ── Generic recursive IRF simulator ───────────────────────────────────────
def simulate_irf(ar_coefs, shock_coefs, shock_size, n=N_IRF):
    """
    Simulate ARDL impulse response to a one-time shock.

    ar_coefs    : array of AR coefficients [β₁, β₂, ..., βₚ]
    shock_coefs : array of distributed lag coefficients [γ₀, γ₁, ..., γₛ]
    shock_size  : scalar magnitude of the impulse at t=0
    """
    pi = np.zeros(n)
    for t in range(n):
        for j, g in enumerate(shock_coefs):
            if t == j:
                pi[t] += g * shock_size
        for i, b in enumerate(ar_coefs):
            if t - (i + 1) >= 0:
                pi[t] += b * pi[t - (i + 1)]
    return pi


# ── Compute IRFs for each scenario ────────────────────────────────────────
irf_shock_labels = [k for k in linked_scenarios if k != "Baseline"]
irf_shock_sizes = [linked_scenarios[k][0] for k in irf_shock_labels]

irf_results = {}

for label, s_brent in zip(irf_shock_labels, irf_shock_sizes):
    s_fao = s_brent * FAO_ELASTICITY
    s_neer = s_brent * NEER_ELASTICITY

    irf_results[label] = {
        "oil": simulate_irf(ar_fs, g_oil, s_brent),
        "neer": simulate_irf(ar_fs, g_neer, s_neer),
        "food": simulate_irf(ar_fs_food, g_fao, s_fao) * FOOD_NAB_WEIGHT,
    }


# ── Shock persistence paths (AR(1) decay) ─────────────────────────────────
def ar1_path(shock_size, rho, n=N_IRF):
    return np.array([shock_size * (rho**t) for t in range(n)])


shock_paths = {
    label: {
        "brent": ar1_path(linked_scenarios[label][0], rho_brent),
        "fao": ar1_path(linked_scenarios[label][0] * FAO_ELASTICITY, rho_fao),
        "neer": ar1_path(linked_scenarios[label][0] * NEER_ELASTICITY, rho_neer),
    }
    for label in irf_shock_labels
}

print("IRF simulation complete.")
print(f"  Horizons : {N_IRF} quarters")
print(f"  Scenarios: {irf_shock_labels}")
print(f"\n  Full-sample AR coefficients (Eq.1): {ar_fs.round(4)}")
print(f"  Oil DL coefficients                : {g_oil.round(4)}")
print(f"  NEER DL coefficients               : {g_neer.round(4)}")
print(f"\n  Full-sample AR coefficients (Eq.5): {ar_fs_food.round(4)}")
print(f"  FAO DL coefficients                : {g_fao.round(4)}")

In [ ]:
config = {
    "title": "Figure 5: External channels, not domestic pressure, drove the swings in headline inflation",
    "subtitle": (
        "Decomposition of South African headline CPI, % yoy, into DIPI and the FX, oil and food contributions. Stacked contributions sum to the headline line by construction"
    ),
    "source": "Source: Authors' calculations",
    "y_label": "YoY Change (%)",
    "figsize": (14, 7),
    "fontscale": 1.0,
    "colors": {
        "dipi": "dodgerblue",
        "fx": "darkgoldenrod",
        "oil": "#6A51A3",
        "food": "forestgreen",
        "cpi": "black",
    },
    "labels": {
        "dipi": "DIPI",
        "fx": "FX",
        "oil": "Oil",
        "food": "Food",
        "cpi": "Headline CPI",
    },
}

fs = config["fontscale"]

# --- Data preparation ---
plot_df = yoy.dropna().copy()
plot_dates = plot_df.index

# Stacked bar components — order determines visual stack from bottom up
components = [
    ("dipi_yoy", config["colors"]["dipi"], config["labels"]["dipi"]),
    ("neer_effect_yoy", config["colors"]["fx"], config["labels"]["fx"]),
    ("oil_effect_yoy", config["colors"]["oil"], config["labels"]["oil"]),
    ("food_effect_yoy", config["colors"]["food"], config["labels"]["food"]),
]

fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Stacked bars — positive and negative handled separately ---
bar_width = 60  # days — controls visual bar width on date axis

pos_bottom = np.zeros(len(plot_df))
neg_bottom = np.zeros(len(plot_df))

for col, color, label in components:
    values = plot_df[col].values

    pos_vals = np.where(values >= 0, values, 0)
    neg_vals = np.where(values < 0, values, 0)

    ax.bar(
        plot_dates,
        pos_vals,
        bottom=pos_bottom,
        width=bar_width,
        color=color,
        label=label,
        align="center",
        zorder=2,
    )
    ax.bar(
        plot_dates,
        neg_vals,
        bottom=neg_bottom,
        width=bar_width,
        color=color,
        align="center",
        zorder=2,
    )

    pos_bottom += pos_vals
    neg_bottom += neg_vals

# --- CPI line overlay ---
ax.plot(
    plot_dates,
    plot_df["headline_yoy"],
    color=config["colors"]["cpi"],
    linewidth=2.0,
    label=config["labels"]["cpi"],
    zorder=3,
)

# --- Axes ---
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)

# --- Grid and reference lines ---
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.axhline(0, color="black", linewidth=0.8, zorder=1)

# --- Y-axis limits ---
all_vals = plot_df[
    ["headline_yoy", "dipi_yoy", "neer_effect_yoy", "oil_effect_yoy", "food_effect_yoy"]
]
y_min = all_vals.min().min()
y_max = all_vals.max().max()
padding = 0.75
ax.set_ylim(y_min - padding, y_max + padding)

ax.spines[["top", "right"]].set_visible(False)

# --- Legend ---
ax.legend(
    fontsize=10 * fs,
    loc="upper right",
    framealpha=0.9,
    edgecolor="none",
    ncol=5,
)

plt.show()

In [ ]:
config = {
    "title": "Figure 7: The external drag on headline empties out by end-2027",
    "subtitle": (
        "Historical and projected external contributions to headline CPI, % yoy, on a hold-constant basis all future FX, oil and food changes set to zero from 2026 Q2). Projection region shaded"
    ),
    "source": "Source: Authors' calculations",
    "y_label": "Contribution to headline CPI (pp, yoy)",
    "figsize": (14, 7),
    "fontscale": 1.0,
    "colors": {
        "fx": "darkgoldenrod",
        "oil": "#6A51A3",
        "food": "forestgreen",
        "total contribution": "black",
    },
    "labels": {
        "fx": "FX",
        "oil": "Oil",
        "food": "Food",
        "total contribution": "Total Contribution",
    },
}

fs = config["fontscale"]

# --- Data preparation ---
# Full series including projection, starting 2015 Q1
plot_df = yoy_full.dropna(subset=["neer_effect_yoy"]).copy()
plot_df = plot_df.loc["2015-03-31":]
plot_df["total_external"] = (
    plot_df["neer_effect_yoy"] + plot_df["oil_effect_yoy"] + plot_df["food_effect_yoy"]
)

plot_dates = plot_df.index
proj_start = pd.Timestamp("2026-03-31")
x_max = plot_dates[-1] + pd.DateOffset(months=2)

fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Projection region shading ---
ax.axvspan(proj_start, x_max, color="gray", alpha=0.12, zorder=0)

# --- Stacked bars — positive and negative tracked separately ---
bar_width = 60  # days

components = [
    ("neer_effect_yoy", config["colors"]["fx"], config["labels"]["fx"]),
    ("oil_effect_yoy", config["colors"]["oil"], config["labels"]["oil"]),
    ("food_effect_yoy", config["colors"]["food"], config["labels"]["food"]),
]

pos_bottom = np.zeros(len(plot_df))
neg_bottom = np.zeros(len(plot_df))

for col, color, label in components:
    values = plot_df[col].values

    pos_vals = np.where(values >= 0, values, 0)
    neg_vals = np.where(values < 0, values, 0)

    ax.bar(
        plot_dates,
        pos_vals,
        bottom=pos_bottom,
        width=bar_width,
        color=color,
        label=label,
        align="center",
        zorder=2,
    )
    ax.bar(
        plot_dates,
        neg_vals,
        bottom=neg_bottom,
        width=bar_width,
        color=color,
        align="center",
        zorder=2,
    )

    pos_bottom += pos_vals
    neg_bottom += neg_vals

# --- Total line ---
ax.plot(
    plot_dates,
    plot_df["total_external"],
    color=config["colors"]["total contribution"],
    linewidth=2.0,
    label=config["labels"]["total contribution"],
    zorder=3,
)

# --- Projection boundary ---
ax.axvline(x=proj_start, color="black", linewidth=1.0, linestyle="--", zorder=4)

# Projection annotation — matches GS style
y_ann = plot_df["total_external"].max() * 0.6
ax.annotate(
    "Hold-constant\nprojection",
    xy=(proj_start, y_ann),
    xytext=(proj_start + pd.DateOffset(months=4), y_ann),
    fontsize=8.5 * fs,
    color="dimgray",
    style="italic",
    va="center",
)

# --- Axes ---
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)
ax.set_xlim(plot_dates[0] - pd.DateOffset(months=2), x_max)

# --- Grid and reference lines ---
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.axhline(0, color="black", linewidth=0.8, zorder=1)

# --- Y-axis limits ---
y_min = (
    plot_df[["neer_effect_yoy", "oil_effect_yoy", "food_effect_yoy", "total_external"]]
    .min()
    .min()
)
y_max = (
    plot_df[["neer_effect_yoy", "oil_effect_yoy", "food_effect_yoy", "total_external"]]
    .max()
    .max()
)
padding = 0.4
ax.set_ylim(y_min - padding, y_max + padding)

ax.spines[["top", "right"]].set_visible(False)

# --- Legend ---
ax.legend(
    fontsize=10 * fs,
    loc="upper left",
    framealpha=0.9,
    edgecolor="none",
    ncol=5,
)

plt.savefig(
    "../graphics/exhibits/exhibit_07_hold_constant.png", dpi=200, bbox_inches="tight"
)
plt.show()

In [ ]:
config = {
    "title": "Figure 8: A severe oil shock adds ~1pp to the oil channel at peak",
    "subtitle": (
        "Projected oil-channel contribution to headline CPI, % yoy, under baseline and oil-only Brent shock scenarios"
    ),
    "source": "Source: Authors' calculations",
    "y_label": "Contribution to headline CPI (pp, yoy)",
    "figsize": (14, 7),
    "fontscale": 1.0,
}

# Scenario line styling — sequential single-hue ramp (light → dark with severity),
# with distinct linestyles as a secondary encoding
scenario_line_style = {
    "Baseline": dict(color="black", linestyle="-", linewidth=1.5),
    "Mild (+5%)": dict(color="#9C7FD4", linestyle=":", linewidth=2.0),
    "Moderate (+10%)": dict(color="#7A54B8", linestyle="--", linewidth=2.0),
    "Severe (+20%)": dict(color="#45276B", linestyle="-.", linewidth=2.0),
}

OIL_COLOR = "#6A51A3"

fs = config["fontscale"]

# --- Data preparation ---
# Historical oil-channel bars — 2015 Q1 to 2026 Q1 (observed only)
hist_oil = yoy_full.loc[~yoy_full["is_projected"], "oil_effect_yoy"]
hist_oil = hist_oil.loc["2015-03-31":"2026-03-31"]

# Scenario oil-channel lines — anchored at the last observed quarter (2026 Q1)
# so the lines connect to the end of the historical bars
scenario_lines = {
    name: scenario_df[name].loc["2026-03-31":] for name in scenarios.keys()
}

proj_start = pd.Timestamp("2026-03-31")
x_max = pd.Timestamp("2028-01-01")

# Y-range, computed here so the annotation can be placed relative to it
all_vals = pd.concat([hist_oil] + list(scenario_lines.values()))
padding = 0.25
y_min = all_vals.min() - padding
y_max = all_vals.max() + padding

fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Projection region shading ---
ax.axvspan(proj_start, x_max, color="gray", alpha=0.12, zorder=0)

# --- Historical oil-channel bars ---
bar_width = 60

ax.bar(
    hist_oil.index,
    hist_oil.values,
    width=bar_width,
    color=OIL_COLOR,
    label="Oil contribution (historical)",
    align="center",
    zorder=2,
)

# --- Scenario lines in projection area ---
for name, series in scenario_lines.items():
    ax.plot(
        series.index, series.values, label=name, zorder=3, **scenario_line_style[name]
    )

# --- Projection boundary ---
ax.axvline(x=proj_start, color="black", linewidth=1.0, linestyle="--", zorder=4)

y_ann = y_max - 0.17  # just under the top of the axis, clear of all four lines
ax.annotate(
    "Projected impact\nof shocks",
    xy=(proj_start, y_ann),
    xytext=(proj_start + pd.DateOffset(months=1), y_ann),
    fontsize=8.5 * fs,
    color="dimgray",
    style="italic",
    va="center",
)

# --- Axes ---
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)
ax.set_xlim(pd.Timestamp("2015-01-01"), x_max)

# --- Grid and reference lines ---
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.axhline(0, color="black", linewidth=0.8, zorder=1)

# --- Y-axis limits ---
ax.set_ylim(y_min, y_max)

ax.spines[["top", "right"]].set_visible(False)

# --- Legend ---
ax.legend(
    fontsize=9.5 * fs,
    loc="upper left",
    framealpha=0.9,
    edgecolor="none",
    ncol=1,
)

plt.show()

In [ ]:
# Line styling for the linked-scenario chart (kept for record; excluded from the article)
scenario_colors = {name: "dimgrey" for name in scenarios}
scenario_styles = {
    "Baseline": "-",
    "Mild (+5%)": ":",
    "Moderate (+10%)": "--",
    "Severe (+20%)": "-.",
}
scenario_widths = {name: (1.5 if name == "Baseline" else 2.0) for name in scenarios}

config_linked = {
    "title": "Figure 8: Projected Contributions to Headline CPI — Linked Shock Scenarios",
    "subtitle": (
        "Historical: FX, Oil and Food contributions  |  "
        "Projection: Total external contribution under linked shock scenarios\n"
        "Oil, FAO and NEER shocked simultaneously — FAO scaled at 0.45× and NEER at −0.54× Brent shock"
    ),
    "source": "Source: Author's own calculations",
    "y_label": "YoY Change (%)",
    "figsize": (14, 7),
    "fontscale": 1.0,
    "colors": {
        "fx": "darkgoldenrod",
        "oil": "darkgray",
        "food": "forestgreen",
        "total": "black",
    },
    "labels": {
        "fx": "FX",
        "oil": "Oil",
        "food": "Food",
        "total": "Total",
    },
}

fs = config_linked["fontscale"]

# ── Data preparation ──────────────────────────────────────────────────────────

# Historical bars — identical to oil-only chart
hist_df = yoy_full.dropna(subset=["neer_effect_yoy"]).copy()
hist_df = hist_df.loc["2015-03-31":"2026-03-31"]
hist_df = hist_df[~hist_df["is_projected"]]
hist_dates = hist_df.index

# Projection rows
proj_df = yoy_full.loc[yoy_full["is_projected"]].copy()

# Anchor point at 2026 Q1 — connects historical bars to projection lines
anchor_date = pd.Timestamp("2026-03-31")
anchor_total = (
    yoy_full.loc[anchor_date, "neer_effect_yoy"]
    + yoy_full.loc[anchor_date, "oil_effect_yoy"]
    + yoy_full.loc[anchor_date, "food_effect_yoy"]
)
proj_start = pd.Timestamp("2026-03-31")

# ── Figure ────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=config_linked["figsize"])
fig.subplots_adjust(top=0.75, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.910,
    config_linked["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.05,
    0.870,
    config_linked["subtitle"],
    transform=fig.transFigure,
    fontsize=9.5 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config_linked["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# ── Historical stacked bars ───────────────────────────────────────────────────

bar_width = 60
components = [
    ("neer_effect_yoy", config_linked["colors"]["fx"], config_linked["labels"]["fx"]),
    ("oil_effect_yoy", config_linked["colors"]["oil"], config_linked["labels"]["oil"]),
    (
        "food_effect_yoy",
        config_linked["colors"]["food"],
        config_linked["labels"]["food"],
    ),
]

pos_bottom = np.zeros(len(hist_df))
neg_bottom = np.zeros(len(hist_df))

for col, color, label in components:
    values = hist_df[col].values
    pos_vals = np.where(values >= 0, values, 0)
    neg_vals = np.where(values < 0, values, 0)

    ax.bar(
        hist_dates,
        pos_vals,
        bottom=pos_bottom,
        width=bar_width,
        color=color,
        label=label,
        align="center",
        zorder=2,
    )
    ax.bar(
        hist_dates,
        neg_vals,
        bottom=neg_bottom,
        width=bar_width,
        color=color,
        align="center",
        zorder=2,
    )

    pos_bottom += pos_vals
    neg_bottom += neg_vals

# ── Historical Total line ─────────────────────────────────────────────────────

hist_total = (
    hist_df["neer_effect_yoy"] + hist_df["oil_effect_yoy"] + hist_df["food_effect_yoy"]
)

ax.plot(
    hist_dates,
    hist_total,
    color="black",
    linestyle="-",
    linewidth=1.5,
    label="Total",
    zorder=3,
)

# ── Linked scenario projection lines ─────────────────────────────────────────

for name in linked_scenarios.keys():
    # Use linked_total_df for projection — all three channels scenario-specific
    proj_total = linked_total_df.loc[proj_df.index, name]

    # Prepend anchor point so line connects cleanly to historical bars
    line_dates = pd.Index([anchor_date]).append(proj_df.index)
    line_values = pd.concat(
        [pd.Series([anchor_total], index=[anchor_date]), proj_total]
    )

    ax.plot(
        line_dates,
        line_values,
        color=scenario_colors[name],
        linestyle=scenario_styles[name],
        linewidth=scenario_widths[name],
        label=name,
        zorder=3,
    )

# ── Projection boundary ───────────────────────────────────────────────────────

ax.axvline(x=proj_start, color="black", linewidth=1.0, linestyle="--", zorder=4)

proj_vals = linked_total_df.loc[proj_df.index]
y_ann = proj_vals.max().max() * 1.75

ax.annotate(
    "Projected impact\nof shocks",
    xy=(proj_start, y_ann),
    xytext=(proj_start + pd.DateOffset(months=4), y_ann),
    fontsize=8.5 * fs,
    color="dimgray",
    style="italic",
    va="center",
)

# ── Axes ──────────────────────────────────────────────────────────────────────

ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config_linked["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)

# ── Grid and reference lines ──────────────────────────────────────────────────

ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.axhline(0, color="black", linewidth=0.8, zorder=1)

# ── Y-axis limits ─────────────────────────────────────────────────────────────

all_hist = hist_df[["neer_effect_yoy", "oil_effect_yoy", "food_effect_yoy"]]
all_proj = linked_total_df
y_min = min(all_hist.min().min(), all_proj.min().min())
y_max = max(all_hist.max().max(), all_proj.max().max())
padding = 0.4
ax.set_ylim(y_min - padding, y_max + padding)

ax.spines[["top", "right"]].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────────

ax.legend(
    fontsize=9.5 * fs,
    loc="upper left",
    framealpha=0.9,
    edgecolor="none",
    ncol=4,
)

plt.savefig("chart_linked_scenarios_total.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── IRF chart setup ───────────────────────────────────────────────────────

N_IRF = len(irf_brent_last)  # derive from actual IRF length, not hardcoded
horizons = np.arange(N_IRF)

irf_shock_names = ["Mild (+5%)", "Moderate (+10%)", "Severe (+20%)"]
irf_shock_sizes = [5.0, 10.0, 20.0]

irf_cfg = {
    "colors": ["#1D9E75", "#185FA5", "#993C1D"],
    "linestyles": ["-", "--", "-."],
    "linewidths": [1.8, 1.8, 1.8],
}

In [ ]:
# All-channel IRF — linked scenarios
# FX and food shocks derived from Brent via existing elasticities,
# consistent with the linked scenario construction above.

channel_specs = [
    {
        "ax_title": "Oil channel",
        "irf": irf_brent_last,
        "scale": lambda s: s,
        "ylabel": "pp — quarterly CPI",
    },
    {
        "ax_title": "FX channel",
        "irf": irf_neer_last,
        "scale": lambda s: s * NEER_ELASTICITY,  # +5% Brent → −2.7% NEER
        "ylabel": "pp — quarterly CPI",
    },
    {
        "ax_title": "Food channel",
        "irf": irf_fao_last,
        "scale": lambda s: s * FAO_ELASTICITY * FOOD_NAB_WEIGHT,
        "ylabel": "pp — quarterly CPI",
    },
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fig.add_artist(
    plt.Line2D(
        [0.05, 0.95],
        [0.97, 0.97],
        transform=fig.transFigure,
        color="black",
        linewidth=1.2,
    )
)

for ax, ch in zip(axes, channel_specs):
    for i, (name, shock) in enumerate(zip(irf_shock_names, irf_shock_sizes)):
        response = np.asarray(ch["irf"][:N_IRF]) * ch["scale"](shock)
        ax.plot(
            horizons,
            response,
            color=irf_cfg["colors"][i],
            linestyle=irf_cfg["linestyles"][i],
            linewidth=irf_cfg["linewidths"][i],
            label=name,
        )
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(ch["ax_title"], fontsize=11, fontweight="bold", pad=6)
    ax.set_xlabel("Quarters after shock", fontsize=9)
    ax.set_ylabel(ch["ylabel"], fontsize=9)
    ax.set_xlim(0, N_IRF - 1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)

fig.text(
    0.05,
    0.93,
    "External Channel Impulse Responses — Linked Scenarios",
    fontsize=13,
    fontweight="bold",
    ha="left",
)
fig.text(
    0.05,
    0.89,
    "FAO scaled at 0.45× and NEER at −0.54× Brent shock  |  "
    "Quarterly headline CPI contributions, pp  |  Last rolling window coefficients (2026 Q1)",
    fontsize=9,
    style="italic",
    ha="left",
    color="#555555",
)
fig.text(
    0.05,
    0.02,
    "Source: Author's own calculations",
    fontsize=8,
    color="#888888",
    ha="left",
)
fig.add_artist(
    plt.Line2D(
        [0.05, 0.95],
        [0.04, 0.04],
        transform=fig.transFigure,
        color="black",
        linewidth=0.6,
    )
)

plt.tight_layout(rect=[0, 0.06, 1, 0.88])
plt.show()

In [ ]:
# Actual 2026 Q1 shock trace
# Uses observed quarterly changes in Brent, FAO and NEER from the final
# estimation quarter as a one-time impulse. Shows how the specific shocks
# already in the system at 2026 Q1 decay forward in isolation.

actual_brent = df.loc[last_t, "brent_qoq"]
actual_fao = df.loc[last_t, "ffpi_qoq"]
actual_neer = df.loc[last_t, "neer_qoq"]

irf_oil_actual = np.asarray(irf_brent_last[:N_IRF]) * actual_brent
irf_neer_actual = np.asarray(irf_neer_last[:N_IRF]) * actual_neer
irf_food_actual = np.asarray(irf_fao_last[:N_IRF]) * actual_fao * FOOD_NAB_WEIGHT
irf_total_actual = irf_oil_actual + irf_neer_actual + irf_food_actual

trace_specs = [
    (irf_oil_actual, "Oil", "#888888", "-", 1.5),
    (irf_neer_actual, "FX", "#BA7517", "--", 1.5),
    (irf_food_actual, "Food", "#3B6D11", "-.", 1.5),
    (irf_total_actual, "Total", "#000000", "-", 2.2),
]

fig, ax = plt.subplots(figsize=(10, 5))

fig.add_artist(
    plt.Line2D(
        [0.05, 0.95],
        [0.97, 0.97],
        transform=fig.transFigure,
        color="black",
        linewidth=1.2,
    )
)

for data, label, color, ls, lw in trace_specs:
    ax.plot(horizons, data, color=color, linestyle=ls, linewidth=lw, label=label)

ax.axhline(0, color="black", linewidth=0.6)
ax.set_xlabel("Quarters after shock", fontsize=10)
ax.set_ylabel("pp contribution to quarterly headline CPI", fontsize=10)
ax.set_xlim(0, N_IRF - 1)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=9)

fig.text(
    0.05,
    0.93,
    "Estimated CPI Impact — 2026 Q1 Observed Shocks",
    fontsize=13,
    fontweight="bold",
    ha="left",
)
fig.text(
    0.05,
    0.89,
    f"One-time impulse from actual 2026 Q1 changes: "
    f"Brent {actual_brent:+.1f}%,  FAO {actual_fao:+.1f}%,  "
    f"NEER {actual_neer:+.1f}%  |  Quarterly CPI contributions, pp",
    fontsize=9,
    style="italic",
    ha="left",
    color="#555555",
)
fig.text(
    0.05,
    0.02,
    "Source: Author's own calculations",
    fontsize=8,
    color="#888888",
    ha="left",
)
fig.add_artist(
    plt.Line2D(
        [0.05, 0.95],
        [0.04, 0.04],
        transform=fig.transFigure,
        color="black",
        linewidth=0.6,
    )
)

plt.tight_layout(rect=[0, 0.06, 1, 0.88])
plt.show()

In [ ]:
# IRF charts

# Horizon axis derived from the full-sample IRF length so this chart is run-order proof.
# (An earlier rolling-window setup leaves the shared `horizons`/`N_IRF` at length 12,
#  whereas these charts plot the length-24 full-sample `irf_results`.)
N_IRF = len(next(iter(irf_results.values()))["oil"])
horizons = np.arange(N_IRF)

colors = ["#1D9E75", "#185FA5", "#993C1D"]
linestyles = ["-", "--", "-."]
lw = 1.8

channel_specs = [
    {
        "title": "Oil channel",
        "irf_key": "oil",
        "path_key": "brent",
        "path_label": "Brent oil price (% dev. from baseline)",
        "rho": rho_brent,
        "rho_label": "Brent",
    },
    {
        "title": "FX channel",
        "irf_key": "neer",
        "path_key": "neer",
        "path_label": "NEER (% dev. from baseline)",
        "rho": rho_neer,
        "rho_label": "NEER",
    },
    {
        "title": "Food channel",
        "irf_key": "food",
        "path_key": "fao",
        "path_label": "FAO index (% dev. from baseline)",
        "rho": rho_fao,
        "rho_label": "FAO",
    },
]

for ch in channel_specs:
    fig, (ax_irf, ax_path) = plt.subplots(
        2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 2], "hspace": 0.45}
    )

    # Top rule
    fig.add_artist(
        plt.Line2D(
            [0.05, 0.95],
            [0.97, 0.97],
            transform=fig.transFigure,
            color="black",
            linewidth=1.2,
        )
    )

    # ── Top panel: CPI contribution IRF ───────────────────────────────────
    for i, label in enumerate(irf_shock_labels):
        ax_irf.plot(
            horizons,
            irf_results[label][ch["irf_key"]],
            color=colors[i],
            linestyle=linestyles[i],
            linewidth=lw,
            label=label,
            marker="o",
            markersize=2.5,
        )

    ax_irf.axhline(0, color="black", linewidth=0.6)
    ax_irf.set_ylabel("pp contribution to quarterly CPI", fontsize=9)
    ax_irf.set_xlabel("Quarters after shock", fontsize=9)
    ax_irf.set_xlim(0, N_IRF - 1)
    ax_irf.spines[["top", "right"]].set_visible(False)
    ax_irf.legend(frameon=False, fontsize=9)
    ax_irf.set_title(
        f"{ch['title']} — CPI impulse response", fontsize=11, fontweight="bold", pad=6
    )

    # ── Bottom panel: shock persistence path ──────────────────────────────
    for i, label in enumerate(irf_shock_labels):
        ax_path.plot(
            horizons,
            shock_paths[label][ch["path_key"]],
            color=colors[i],
            linestyle="--",
            linewidth=lw,
            label=label,
            marker="o",
            markersize=2.5,
        )

    ax_path.axhline(0, color="black", linewidth=0.6)
    ax_path.set_ylabel("% deviation from baseline", fontsize=9)
    ax_path.set_xlabel("Quarters after shock", fontsize=9)
    ax_path.set_xlim(0, N_IRF - 1)
    ax_path.spines[["top", "right"]].set_visible(False)
    ax_path.set_title(ch["path_label"], fontsize=10, pad=6)
    ax_path.text(
        0.99,
        0.95,
        f"ρ = {ch['rho']:.3f} (estimated AR(1) on quarterly {ch['rho_label']})",
        transform=ax_path.transAxes,
        fontsize=8,
        color="#888888",
        ha="right",
        va="top",
    )

    # Figure-level titles and footer
    fig.text(
        0.05,
        0.93,
        f"{ch['title']} — Impulse Response",
        fontsize=13,
        fontweight="bold",
        ha="left",
    )
    fig.text(
        0.05,
        0.895,
        "Full-sample ARDL coefficients  |  Pure one-time impulse  |  "
        "FX and Food shocks scaled via linked-scenario elasticities",
        fontsize=9,
        style="italic",
        ha="left",
        color="#555555",
    )
    fig.text(
        0.05,
        0.02,
        "Source: Author's own calculations",
        fontsize=8,
        color="#888888",
        ha="left",
    )
    fig.add_artist(
        plt.Line2D(
            [0.05, 0.95],
            [0.04, 0.04],
            transform=fig.transFigure,
            color="black",
            linewidth=0.6,
        )
    )

    plt.tight_layout(rect=[0, 0.06, 1, 0.89])
    plt.show()

In [ ]:
# Exhibit A1 — Oil channel IRF: one-time +10% Brent shock
# Own-channel standardised shock only (no cross-channel elasticities), so the
# chart is consistent with the oil-only scenario design in the article.
# Full-sample Eq. 1 (headline ARDL) coefficients; run-order proof — everything is re-derived
# here from the fitted model, independent of earlier scenario sections.

N_IRF = 24
horizons = np.arange(N_IRF)
SHOCK = 10.0  # one-time +10% q/q move in Brent

b_eq1 = res_emp.params
ar_fs = np.array([b_eq1[f"headline_qoq.L{i}"] for i in range(1, 5)])
g_oil = np.array([b_eq1["brent_qoq.L0"], b_eq1["brent_qoq.L1"]])

irf_ch = simulate_irf(ar_fs, g_oil, SHOCK, n=N_IRF)
cum_ch = np.cumsum(irf_ch)
lr_ch = SHOCK * g_oil.sum() / (1 - ar_fs.sum())  # = 10 x OPT

config = {
    "figsize": (16, 6),
    "fontscale": 1.0,
    "title": "Exhibit A1: The oil channel is front-loaded, with most of the pass-through inside a year",
    "subtitle": "Response of quarterly headline CPI inflation to a one-time +10% q/q Brent shock  |  Full-sample Eq. 1 coefficients  |  FX and food channels held constant",
    "source": "Source: Authors' calculations",
    "wspace": 0.35,
    "color": "#6A51A3",
}

fs = config["fontscale"]

fig, (ax_irf, ax_cum) = plt.subplots(1, 2, figsize=config["figsize"])
fig.subplots_adjust(
    top=0.78, bottom=0.18, left=0.06, right=0.98, wspace=config["wspace"]
)

fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# -- Left panel: quarterly impulse response (plotted at quarterly points, no
#    interpolation, so the printed values can be read off the chart) ---------
ax_irf.plot(
    horizons, irf_ch, color=config["color"], linewidth=1.8, marker="o", markersize=3.5
)
ax_irf.axhline(0, color="black", linewidth=0.6)
ax_irf.set_title(
    "Quarterly CPI contribution", fontsize=11 * fs, fontweight="bold", pad=6, loc="left"
)
ax_irf.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_irf.set_ylabel("pp contribution to quarterly CPI", fontsize=10 * fs)
ax_irf.set_xlim(0, N_IRF - 1)
ax_irf.tick_params(labelsize=9 * fs)
ax_irf.grid(axis="y", linestyle="--", alpha=0.5)
ax_irf.spines[["top", "right"]].set_visible(False)

# -- Right panel: cumulative response vs analytic long-run pass-through ------
ax_cum.plot(
    horizons,
    cum_ch,
    color=config["color"],
    linewidth=1.8,
    marker="o",
    markersize=3.5,
    label="Cumulative response",
)
ax_cum.axhline(
    lr_ch,
    color="black",
    linewidth=1.0,
    linestyle="--",
    label=f"Long-run (10 x OPT) = {lr_ch:.2f}pp",
)
ax_cum.axhline(0, color="black", linewidth=0.6)
ax_cum.set_title(
    "Cumulative CPI contribution",
    fontsize=11 * fs,
    fontweight="bold",
    pad=6,
    loc="left",
)
ax_cum.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_cum.set_ylabel("Cumulative pp contribution", fontsize=10 * fs)
ax_cum.set_xlim(0, N_IRF - 1)
ax_cum.tick_params(labelsize=9 * fs)
ax_cum.legend(frameon=False, fontsize=9 * fs)
ax_cum.grid(axis="y", linestyle="--", alpha=0.5)
ax_cum.spines[["top", "right"]].set_visible(False)

import os

os.makedirs("../graphics/exhibits", exist_ok=True)
plt.savefig("../graphics/exhibits/exhibit_A1_irf_oil.png", dpi=150, bbox_inches="tight")
plt.show()

# -- Verification: simulated cumulative response must converge on the
#    closed-form long-run scalar ---------------------------------------------
peak_q = int(np.argmax(np.abs(irf_ch)))
print(f"Oil channel | shock {SHOCK:+.0f}%")
print(f"  peak response      : {irf_ch[peak_q]:+.4f}pp in quarter {peak_q}")
print(
    f"  half-life          : cumulative reaches 50% of long-run by quarter "
    f"{int(np.argmax(cum_ch / lr_ch >= 0.5))}, 90% by quarter {int(np.argmax(cum_ch / lr_ch >= 0.9))}"
)
print(f"  cumulative @ q{N_IRF - 1}  : {cum_ch[-1]:+.4f}pp")
print(f"  analytic long-run  : {lr_ch:+.4f}pp  (gap {abs(cum_ch[-1] - lr_ch):.5f}pp)")

In [ ]:
# Exhibit A2 — FX channel IRF: one-time 10% rand depreciation
# Own-channel standardised shock only (no cross-channel elasticities), so the
# chart is consistent with the oil-only scenario design in the article.
# Full-sample Eq. 1 (headline ARDL) coefficients; run-order proof — everything is re-derived
# here from the fitted model, independent of earlier scenario sections.

N_IRF = 24
horizons = np.arange(N_IRF)
SHOCK = (
    -10.0
)  # one-time 10% depreciation = -10% q/q NEER (appreciation-positive convention)

b_eq1 = res_emp.params
ar_fs = np.array([b_eq1[f"headline_qoq.L{i}"] for i in range(1, 5)])
g_neer = np.array([b_eq1["neer_qoq.L0"], b_eq1["neer_qoq.L1"], b_eq1["neer_qoq.L2"]])

irf_ch = simulate_irf(ar_fs, g_neer, SHOCK, n=N_IRF)
cum_ch = np.cumsum(irf_ch)
lr_ch = SHOCK * g_neer.sum() / (1 - ar_fs.sum())  # = 10 x ERPT

config = {
    "figsize": (16, 6),
    "fontscale": 1.0,
    "title": "Exhibit A2: A 10% rand depreciation adds ~1.8pp to headline in the long run — the largest channel",
    "subtitle": "Response of quarterly headline CPI inflation to a one-time 10% rand depreciation (−10% q/q NEER)  |  Full-sample Eq. 1 coefficients  |  Oil and food channels held constant",
    "source": "Source: Authors' calculations",
    "wspace": 0.35,
    "color": "dodgerblue",
}

fs = config["fontscale"]

fig, (ax_irf, ax_cum) = plt.subplots(1, 2, figsize=config["figsize"])
fig.subplots_adjust(
    top=0.78, bottom=0.18, left=0.06, right=0.98, wspace=config["wspace"]
)

fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# -- Left panel: quarterly impulse response (plotted at quarterly points, no
#    interpolation, so the printed values can be read off the chart) ---------
ax_irf.plot(
    horizons, irf_ch, color=config["color"], linewidth=1.8, marker="o", markersize=3.5
)
ax_irf.axhline(0, color="black", linewidth=0.6)
ax_irf.set_title(
    "Quarterly CPI contribution", fontsize=11 * fs, fontweight="bold", pad=6, loc="left"
)
ax_irf.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_irf.set_ylabel("pp contribution to quarterly CPI", fontsize=10 * fs)
ax_irf.set_xlim(0, N_IRF - 1)
ax_irf.tick_params(labelsize=9 * fs)
ax_irf.grid(axis="y", linestyle="--", alpha=0.5)
ax_irf.spines[["top", "right"]].set_visible(False)

# -- Right panel: cumulative response vs analytic long-run pass-through ------
ax_cum.plot(
    horizons,
    cum_ch,
    color=config["color"],
    linewidth=1.8,
    marker="o",
    markersize=3.5,
    label="Cumulative response",
)
ax_cum.axhline(
    lr_ch,
    color="black",
    linewidth=1.0,
    linestyle="--",
    label=f"Long-run (10 x ERPT) = {lr_ch:.2f}pp",
)
ax_cum.axhline(0, color="black", linewidth=0.6)
ax_cum.set_title(
    "Cumulative CPI contribution",
    fontsize=11 * fs,
    fontweight="bold",
    pad=6,
    loc="left",
)
ax_cum.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_cum.set_ylabel("Cumulative pp contribution", fontsize=10 * fs)
ax_cum.set_xlim(0, N_IRF - 1)
ax_cum.tick_params(labelsize=9 * fs)
ax_cum.legend(frameon=False, fontsize=9 * fs)
ax_cum.grid(axis="y", linestyle="--", alpha=0.5)
ax_cum.spines[["top", "right"]].set_visible(False)

import os

os.makedirs("../graphics/exhibits", exist_ok=True)
plt.savefig("../graphics/exhibits/exhibit_A2_irf_fx.png", dpi=150, bbox_inches="tight")
plt.show()

# -- Verification: simulated cumulative response must converge on the
#    closed-form long-run scalar ---------------------------------------------
peak_q = int(np.argmax(np.abs(irf_ch)))
print(f"FX channel | shock {SHOCK:+.0f}%")
print(f"  peak response      : {irf_ch[peak_q]:+.4f}pp in quarter {peak_q}")
print(
    f"  half-life          : cumulative reaches 50% of long-run by quarter "
    f"{int(np.argmax(cum_ch / lr_ch >= 0.5))}, 90% by quarter {int(np.argmax(cum_ch / lr_ch >= 0.9))}"
)
print(f"  cumulative @ q{N_IRF - 1}  : {cum_ch[-1]:+.4f}pp")
print(f"  analytic long-run  : {lr_ch:+.4f}pp  (gap {abs(cum_ch[-1] - lr_ch):.5f}pp)")

In [ ]:
# Exhibit A3 — Food channel IRF: one-time +10% FAO food-price shock
# Own-channel standardised shock only (no cross-channel elasticities), so the
# chart is consistent with the oil-only scenario design in the article.
# Full-sample Eq. 5 (food ARDL) coefficients; run-order proof — everything is re-derived
# here from the fitted model, independent of earlier scenario sections.

N_IRF = 24
horizons = np.arange(N_IRF)
SHOCK = 10.0  # one-time +10% q/q move in the FAO index

b_eq5 = model_food_p4.params
ar_fs_food = np.array([b_eq5[f"food_nab_qoq.L{i}"] for i in range(1, 5)])
g_fao = np.array([b_eq5["ffpi_qoq.L0"], b_eq5["ffpi_qoq.L1"]])

irf_ch = simulate_irf(ar_fs_food, g_fao, SHOCK, n=N_IRF) * FOOD_NAB_WEIGHT
cum_ch = np.cumsum(irf_ch)
lr_ch = (
    SHOCK * g_fao.sum() / (1 - ar_fs_food.sum()) * FOOD_NAB_WEIGHT
)  # = 10 x FPT_headline

config = {
    "figsize": (16, 6),
    "fontscale": 1.0,
    "title": "Exhibit A3: The food channel peaks one quarter after the shock and is the smallest in headline terms",
    "subtitle": "Response of quarterly headline CPI inflation to a one-time +10% q/q FAO shock  |  Full-sample Eq. 5 coefficients, rescaled by the food & NAB basket weight (18.2%)",
    "source": "Source: Authors' calculations",
    "wspace": 0.35,
    "color": "forestgreen",
}

fs = config["fontscale"]

fig, (ax_irf, ax_cum) = plt.subplots(1, 2, figsize=config["figsize"])
fig.subplots_adjust(
    top=0.78, bottom=0.18, left=0.06, right=0.98, wspace=config["wspace"]
)

fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# -- Left panel: quarterly impulse response (plotted at quarterly points, no
#    interpolation, so the printed values can be read off the chart) ---------
ax_irf.plot(
    horizons, irf_ch, color=config["color"], linewidth=1.8, marker="o", markersize=3.5
)
ax_irf.axhline(0, color="black", linewidth=0.6)
ax_irf.set_title(
    "Quarterly CPI contribution", fontsize=11 * fs, fontweight="bold", pad=6, loc="left"
)
ax_irf.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_irf.set_ylabel("pp contribution to quarterly CPI", fontsize=10 * fs)
ax_irf.set_xlim(0, N_IRF - 1)
ax_irf.tick_params(labelsize=9 * fs)
ax_irf.grid(axis="y", linestyle="--", alpha=0.5)
ax_irf.spines[["top", "right"]].set_visible(False)

# -- Right panel: cumulative response vs analytic long-run pass-through ------
ax_cum.plot(
    horizons,
    cum_ch,
    color=config["color"],
    linewidth=1.8,
    marker="o",
    markersize=3.5,
    label="Cumulative response",
)
ax_cum.axhline(
    lr_ch,
    color="black",
    linewidth=1.0,
    linestyle="--",
    label=f"Long-run (10 x FPT_headline) = {lr_ch:.2f}pp",
)
ax_cum.axhline(0, color="black", linewidth=0.6)
ax_cum.set_title(
    "Cumulative CPI contribution",
    fontsize=11 * fs,
    fontweight="bold",
    pad=6,
    loc="left",
)
ax_cum.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_cum.set_ylabel("Cumulative pp contribution", fontsize=10 * fs)
ax_cum.set_xlim(0, N_IRF - 1)
ax_cum.tick_params(labelsize=9 * fs)
ax_cum.legend(frameon=False, fontsize=9 * fs)
ax_cum.grid(axis="y", linestyle="--", alpha=0.5)
ax_cum.spines[["top", "right"]].set_visible(False)

import os

os.makedirs("../graphics/exhibits", exist_ok=True)
plt.savefig(
    "../graphics/exhibits/exhibit_A3_irf_food.png", dpi=150, bbox_inches="tight"
)
plt.show()

# -- Verification: simulated cumulative response must converge on the
#    closed-form long-run scalar ---------------------------------------------
peak_q = int(np.argmax(np.abs(irf_ch)))
print(f"Food channel | shock {SHOCK:+.0f}%")
print(f"  peak response      : {irf_ch[peak_q]:+.4f}pp in quarter {peak_q}")
print(
    f"  half-life          : cumulative reaches 50% of long-run by quarter "
    f"{int(np.argmax(cum_ch / lr_ch >= 0.5))}, 90% by quarter {int(np.argmax(cum_ch / lr_ch >= 0.9))}"
)
print(f"  cumulative @ q{N_IRF - 1}  : {cum_ch[-1]:+.4f}pp")
print(f"  analytic long-run  : {lr_ch:+.4f}pp  (gap {abs(cum_ch[-1] - lr_ch):.5f}pp)")

In [ ]:
# ── IRF Chart: Actual 2026 Q1 shock trace ─────────────────────────────────

actual_brent = df.loc[last_t, "brent_qoq"]
actual_fao = df.loc[last_t, "ffpi_qoq"]
actual_neer = df.loc[last_t, "neer_qoq"]

irf_oil_actual = simulate_irf(ar_fs, g_oil, actual_brent)
irf_neer_actual = simulate_irf(ar_fs, g_neer, actual_neer)
irf_food_actual = simulate_irf(ar_fs_food, g_fao, actual_fao) * FOOD_NAB_WEIGHT
irf_total_actual = irf_oil_actual + irf_neer_actual + irf_food_actual

# Horizon axis derived from the computed IRF length so this chart is run-order proof
# (independent of whatever `N_IRF`/`horizons` an earlier section left behind).
N_IRF = len(irf_total_actual)
horizons = np.arange(N_IRF)

config_irf_actual = {
    "figsize": (16, 6),
    "fontscale": 1.0,
    "title": "Estimated CPI Impact — 2026 Q1 Observed Shocks",
    "subtitle": (
        f"One-time impulse from observed 2026 Q1 changes: "
        f"Oil {actual_brent:+.1f}%,  Food {actual_fao:+.1f}%,  "
        f"FX {actual_neer:+.1f}%  |  Quarterly CPI contributions, pp"
    ),
    "source": "Source: Author's own calculations",
    "wspace": 0.35,
}

trace_specs = [
    (irf_oil_actual, "Oil", "#888888", "-", 1.6),
    (irf_neer_actual, "FX", "#BA7517", "--", 1.6),
    (irf_food_actual, "Food", "#3B6D11", "-.", 1.6),
    (irf_total_actual, "Total", "#000000", "-", 2.2),
]

fs = config_irf_actual["fontscale"]
x_s = np.linspace(0, N_IRF - 1, 300)

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=config_irf_actual["figsize"])
fig.subplots_adjust(
    top=0.78, bottom=0.18, left=0.05, right=0.98, wspace=config_irf_actual["wspace"]
)

fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.905,
    config_irf_actual["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config_irf_actual["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)
fig.text(
    0.05,
    0.07,
    config_irf_actual["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# ── Left panel: individual channel contributions ───────────────────────────
for data, label, color, ls, lw in trace_specs[:-1]:  # Oil, FX, Food only
    spl = make_interp_spline(horizons, data, k=3)
    ax_left.plot(x_s, spl(x_s), color=color, linestyle=ls, linewidth=lw, label=label)

ax_left.axhline(0, color="black", linewidth=0.6)
ax_left.set_title(
    "Channel contributions", fontsize=11 * fs, fontweight="bold", pad=6, loc="left"
)
ax_left.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_left.set_ylabel("pp contribution to quarterly CPI", fontsize=10 * fs)
ax_left.set_xlim(0, N_IRF - 1)
ax_left.tick_params(labelsize=9 * fs)
ax_left.legend(frameon=False, fontsize=9 * fs)
ax_left.grid(axis="y", linestyle="--", alpha=0.5)
ax_left.spines[["top", "right"]].set_visible(False)

# ── Right panel: total contribution ───────────────────────────────────────
data, label, color, ls, lw = trace_specs[-1]  # Total only
spl = make_interp_spline(horizons, data, k=3)
ax_right.plot(x_s, spl(x_s), color=color, linestyle=ls, linewidth=lw, label=label)

ax_right.axhline(0, color="black", linewidth=0.6)
ax_right.set_title(
    "Total external contribution",
    fontsize=11 * fs,
    fontweight="bold",
    pad=6,
    loc="left",
)
ax_right.set_xlabel("Quarters after shock", fontsize=10 * fs)
ax_right.set_ylabel("pp contribution to quarterly CPI", fontsize=10 * fs)
ax_right.set_xlim(0, N_IRF - 1)
ax_right.tick_params(labelsize=9 * fs)
ax_right.legend(frameon=False, fontsize=9 * fs)
ax_right.grid(axis="y", linestyle="--", alpha=0.5)
ax_right.spines[["top", "right"]].set_visible(False)

plt.show()

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
ROLLING_START = "2019-01-01"  # First valid rolling estimate

config = {
    "title": "Figure X: Time-Varying Pass-Through Scalars",
    "subtitle": "Rolling 40-quarter window, ARDL(4,2,1) Eq.1 and ARDL(4,1,1) Eq.5; full-sample backfill pre-2019 Q1",
    "source": "Source: Author's calculations; Stats SA, IMF, UN FAO",
    "x_label": "",
    "y_label_left": "Pass-through scalar (headline CPI)",
    "y_label_right": "Pass-through scalar (food sub-index)",
    "date_interval": 8,  # quarters
    "figsize": (12, 6),
    "fontscale": 1.0,
    "y_padding_left": 0.02,
    "y_padding_right": 0.05,
    "erpt": {
        "label": "ERPT",
        "color": "dodgerblue",
        "linewidth": 2,
        "linestyle": "-",
    },
    "opt": {
        "label": "OPT",
        "color": "firebrick",
        "linewidth": 2,
        "linestyle": "-",
    },
    "fpt_headline": {
        "label": "FPT (headline-scaled)",
        "color": "darkgoldenrod",
        "linewidth": 1.5,
        "linestyle": "-",
    },
    "fpt_food": {
        "label": "FPT (food sub-index), RHS",
        "color": "darkgoldenrod",
        "linewidth": 1.5,
        "linestyle": "--",
    },
}

# ── Plot ──────────────────────────────────────────────────────────
fs = config["fontscale"]

fig, ax_left = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.07, right=0.93)

ax_right = ax_left.twinx()

# --- Top rule line ---
fig.add_artist(
    Line2D(
        [0.0, 1.0],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)

# --- Title & subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config["subtitle"],
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
fig.add_artist(
    Line2D(
        [0.0, 1.0],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)

# --- Source ---
fig.text(
    0.05, 0.07, config["source"], fontsize=9 * fs, va="top", ha="left", color="dimgray"
)

# --- Lines (left axis) ---
ax_left.plot(
    scalars.index,
    scalars["erpt"],
    **{k: config["erpt"][k] for k in ("label", "color", "linewidth", "linestyle")},
)
ax_left.plot(
    scalars.index,
    scalars["opt"],
    **{k: config["opt"][k] for k in ("label", "color", "linewidth", "linestyle")},
)
ax_left.plot(
    scalars.index,
    scalars["fpt_headline"],
    **{
        k: config["fpt_headline"][k]
        for k in ("label", "color", "linewidth", "linestyle")
    },
)

# --- Lines (right axis) ---
ax_right.plot(
    scalars.index,
    scalars["fpt_food"],
    **{k: config["fpt_food"][k] for k in ("label", "color", "linewidth", "linestyle")},
)

# --- Rolling-start demarcation ---
ax_left.axvline(
    pd.Timestamp(ROLLING_START), color="grey", linewidth=0.8, linestyle=":", zorder=1
)
ax_left.text(
    pd.Timestamp(ROLLING_START),
    ax_left.get_ylim()[1],
    "  Rolling estimates begin",
    fontsize=8 * fs,
    va="top",
    ha="left",
    color="grey",
)

# --- Axes formatting ---
ax_left.xaxis.set_major_locator(mdates.MonthLocator(interval=config["date_interval"]))
ax_left.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax_left.get_xticklabels(), rotation=0, ha="right")
ax_left.tick_params(labelsize=9 * fs)
ax_right.tick_params(labelsize=9 * fs)

ax_left.set_ylabel(config["y_label_left"], fontsize=10 * fs)
ax_right.set_ylabel(config["y_label_right"], fontsize=10 * fs)

# --- Y-axis limits ---
p = config["y_padding_left"]
left_vals = pd.concat([scalars["erpt"], scalars["opt"], scalars["fpt_headline"]])
ax_left.set_ylim(left_vals.min() - p, left_vals.max() + p)

p = config["y_padding_right"]
ax_right.set_ylim(scalars["fpt_food"].min() - p, scalars["fpt_food"].max() + p)

# --- Combined legend ---
lines_l, labels_l = ax_left.get_legend_handles_labels()
lines_r, labels_r = ax_right.get_legend_handles_labels()
ax_left.legend(
    lines_l + lines_r, labels_l + labels_r, fontsize=10 * fs, loc="upper left"
)

# --- Grid ---
ax_left.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

In [ ]:
# --- Exhibit 5: Time-varying pass-through elasticities ---

START_DATE = "2019-01-01"  # first valid rolling estimate
END_DATE = "2026-03-31"

config = {
    "title": "Figure 6: Pass-through has drifted, with FX sensitivity easing most recently",
    "subtitle": (
        "Time-varying long-run pass-through elasticities; FX (ERPT), oil (OPT) and food (FPT)"
    ),
    "source": "Source: Authors' calculations",
    "y_label": "Long-run pass-through elasticity",
    "figsize": (12, 6),
    "fontscale": 1.0,
    "series": {
        "erpt": {
            "label": "ERPT",
            "color": "dodgerblue",
            "linewidth": 2.0,
            "linestyle": "-",
        },
        "opt": {"label": "OPT", "color": "#6A51A3", "linewidth": 2.0, "linestyle": "-"},
        "fpt_headline": {
            "label": "FPT",
            "color": "forestgreen",
            "linewidth": 2.0,
            "linestyle": "-",
        },
    },
}

fs = config["fontscale"]

# --- Data preparation ---
sc = scalars.loc[START_DATE:END_DATE, ["erpt", "opt", "fpt_headline"]].copy()

fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.09, right=0.93)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title & subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=9.5 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Lines ---
for col, style in config["series"].items():
    ax.plot(
        sc.index,
        sc[col],
        label=style["label"],
        color=style["color"],
        linewidth=style["linewidth"],
        linestyle=style["linestyle"],
    )

# --- Axes formatting ---
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(3,), interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
ax.tick_params(labelsize=9 * fs)
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.set_xlim(
    sc.index[0] - pd.DateOffset(months=2), sc.index[-1] + pd.DateOffset(months=2)
)

# --- Grid ---
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

# --- Legend ---
ax.legend(fontsize=10 * fs, loc="upper right", framealpha=0.9, edgecolor="none", ncol=3)

plt.show()

In [ ]:
# ── Exhibit 6: Headline, core and DIPI against the target regimes ─────────────
import os

os.makedirs("../graphics/exhibits", exist_ok=True)

config = {
    "title": "Exhibit 6: Domestic pressure held mid-band — and now sits above the new 3% target",
    "subtitle": (
        "Headline CPI, core CPI and domestic pressure (DIPI), % yoy. The 3–6% band is shaded through 2025;\n"
        "the new 2–4% band and 3% point target line apply from 2026 Q1"
    ),
    "source": "Source: Authors' calculations; Stats SA CPI; SARB",
    "y_label": "YoY Change (%)",
    "figsize": (14, 7),
    "fontscale": 1.0,
}

fs = config["fontscale"]

# ── Data preparation ──────────────────────────────────────────────────────────
plot_df = yoy.dropna().copy()
plot_df["core_yoy"] = df["core_qoq"].loc[dates].rolling(4).sum()
plot_dates = plot_df.index

regime_switch = pd.Timestamp("2026-01-01")  # new 2–4% band from 2026 Q1
x_min = plot_dates[0] - pd.DateOffset(months=3)
x_max = pd.Timestamp("2026-09-30")  # margin so the new regime is visible

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.75, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.93, 0.93],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.910,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)
fig.text(
    0.05,
    0.870,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=9.5 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
fig.add_artist(
    Line2D(
        [0.0, 0.98],
        [0.10, 0.10],
        transform=fig.transFigure,
        color="black",
        linewidth=0.8,
    )
)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# ── Target band shading — old regime through 2025, new regime from 2026 Q1 ────
ax.fill_between([x_min, regime_switch], 3, 6, color="gray", alpha=0.12, zorder=0)
ax.fill_between([regime_switch, x_max], 2, 4, color="gray", alpha=0.12, zorder=0)
ax.hlines(
    3, regime_switch, x_max, color="black", linewidth=1.2, linestyle="--", zorder=1
)
ax.axvline(regime_switch, color="gray", linewidth=0.8, linestyle=":", zorder=1)

ax.text(
    plot_dates[2],
    5.75,
    "3–6% target band",
    fontsize=8.5 * fs,
    color="dimgray",
    va="top",
)
ax.annotate(
    "2–4% band,\n3% point target",
    xy=(regime_switch, 4.0),
    xytext=(regime_switch - pd.DateOffset(months=26), 5.1),
    fontsize=8.5 * fs,
    color="dimgray",
    va="center",
    arrowprops=dict(arrowstyle="-", color="dimgray", linewidth=0.7),
)

# ── Lines ─────────────────────────────────────────────────────────────────────
ax.plot(
    plot_dates,
    plot_df["headline_yoy"],
    color="black",
    linewidth=2.0,
    linestyle="-",
    label="Headline CPI",
    zorder=3,
)
ax.plot(
    plot_dates,
    plot_df["core_yoy"],
    color="dimgray",
    linewidth=1.8,
    linestyle="--",
    label="Core CPI",
    zorder=3,
)
ax.plot(
    plot_dates,
    plot_df["dipi_yoy"],
    color="dodgerblue",
    linewidth=2.4,
    linestyle="-",
    label="DIPI",
    zorder=4,
)

# --- Direct labels at line ends, with a minimum vertical gap ---
series_ends = sorted(
    [
        (plot_df["headline_yoy"].iloc[-1], "Headline"),
        (plot_df["core_yoy"].iloc[-1], "Core"),
        (plot_df["dipi_yoy"].iloc[-1], "DIPI"),
    ],
    key=lambda t: t[0],
)
min_gap = 0.45
label_y = []
for i, (y, _) in enumerate(series_ends):
    y_adj = y if i == 0 else max(y, label_y[i - 1] + min_gap)
    label_y.append(y_adj)
for (y, label), y_adj in zip(series_ends, label_y):
    ax.annotate(
        label,
        xy=(plot_dates[-1], y),
        xytext=(plot_dates[-1] + pd.DateOffset(months=1), y_adj),
        fontsize=9 * fs,
        color="dimgray",
        va="center",
    )

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)
ax.set_xlim(x_min, x_max)

# ── Grid and reference lines ──────────────────────────────────────────────────
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

# ── Y-axis limits ─────────────────────────────────────────────────────────────
all_vals = plot_df[["headline_yoy", "core_yoy", "dipi_yoy"]]
padding = 0.6
ax.set_ylim(
    min(all_vals.min().min(), 1.8) - padding, max(all_vals.max().max(), 6.2) + padding
)

ax.spines[["top", "right"]].set_visible(False)

# ── Legend ────────────────────────────────────────────────────────────────────
ax.legend(fontsize=10 * fs, loc="upper right", framealpha=0.9, edgecolor="none", ncol=3)

plt.savefig(
    "../graphics/exhibits/exhibit_06_dipi_vs_target.png", dpi=200, bbox_inches="tight"
)
plt.show()

In [ ]:
config = {
    "title": "Figure 4: Domestic pressure was stable while headline made a round trip",
    "subtitle": "Headline vs Core CPI vs DIPI, yoy % change, quarterly; grey band marks the SARB target range (3–6% through 2025, 2–4% from 2026); dashed line marks the midpoint",
    "source": "Source: Authors' calculations; Stats SA",
    "y_label": "YoY Change (%)",
    "figsize": (14, 7),
    "fontscale": 1.0,
    "regime_break": "2026-01-01",  # 3–6% band before, 2–4% band from here
    "headline": {
        "label": "Headline CPI",
        "color": "black",
        "linewidth": 2,
        "linestyle": "-",
    },
    "core": {
        "label": "Core CPI",
        "color": "dodgerblue",
        "linewidth": 2,
        "linestyle": "-",
    },
    "dipi": {
        "label": "DIPI",
        "color": "dimgray",
        "linewidth": 2,
        "linestyle": "--",
    },
}

fs = config["fontscale"]

# --- Data preparation ---
plot_df = yoy.dropna().copy()
plot_df["core_yoy"] = df["core_qoq"].loc[dates].rolling(4).sum()
plot_dates = plot_df.index

x_min = plot_dates[0] - pd.DateOffset(months=3)
x_max = plot_dates[-1] + pd.DateOffset(months=3)

fig, ax = plt.subplots(figsize=config["figsize"])
fig.subplots_adjust(top=0.78, bottom=0.18, left=0.07, right=0.98)

# --- Top rule line ---
top_line = Line2D(
    [0.0, 0.98], [0.93, 0.93], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(top_line)

# --- Title and subtitle ---
fig.text(
    0.05,
    0.905,
    config["title"],
    transform=fig.transFigure,
    fontsize=13 * fs,
    fontweight="bold",
    va="top",
    ha="left",
)

fig.text(
    0.05,
    0.865,
    config["subtitle"],
    transform=fig.transFigure,
    fontsize=10 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Bottom rule line ---
bottom_line = Line2D(
    [0.0, 0.98], [0.10, 0.10], transform=fig.transFigure, color="black", linewidth=0.8
)
fig.add_artist(bottom_line)

# --- Source text ---
fig.text(
    0.05,
    0.07,
    config["source"],
    transform=fig.transFigure,
    fontsize=9 * fs,
    va="top",
    ha="left",
    color="dimgray",
)

# --- Plot ---
ax.plot(
    plot_dates,
    plot_df["headline_yoy"],
    label=config["headline"]["label"],
    color=config["headline"]["color"],
    linewidth=config["headline"]["linewidth"],
    linestyle=config["headline"]["linestyle"],
    zorder=3,
)

ax.plot(
    plot_dates,
    plot_df["core_yoy"],
    label=config["core"]["label"],
    color=config["core"]["color"],
    linewidth=config["core"]["linewidth"],
    linestyle=config["core"]["linestyle"],
    zorder=3,
)

ax.plot(
    plot_dates,
    plot_df["dipi_yoy"],
    label=config["dipi"]["label"],
    color=config["dipi"]["color"],
    linewidth=config["dipi"]["linewidth"],
    linestyle=config["dipi"]["linestyle"],
    zorder=4,
)

# --- Axes ---
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=0, ha="center")
ax.set_ylabel(config["y_label"], fontsize=10 * fs)
ax.tick_params(labelsize=9 * fs)
ax.set_xlim(x_min, x_max)

# --- Legend ---
ax.legend(fontsize=10 * fs)

# --- Grid ---
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.spines[["top", "right"]].set_visible(False)

# --- Target bands: 3–6% (4.5% midpoint) through 2025, 2–4% (3% midpoint) from 2026 ---
x_left, x_right = ax.get_xlim()
x_break = mdates.date2num(pd.Timestamp(config["regime_break"]))

ax.fill_between([x_left, x_break], 3, 6, color="lightgray", alpha=0.3, zorder=0)
ax.hlines([3, 6], x_left, x_break, color="darkgray", linewidth=0.8)
ax.hlines(4.5, x_left, x_break, color="orangered", linewidth=0.8, linestyle="--")

ax.fill_between([x_break, x_right], 2, 4, color="lightgray", alpha=0.3, zorder=0)
ax.hlines([2, 4], x_break, x_right, color="darkgray", linewidth=0.8)
ax.hlines(3, x_break, x_right, color="orangered", linewidth=0.8, linestyle="--")

ax.set_xlim(x_left, x_right)

# --- Y-axis limits ---
all_vals = plot_df[["headline_yoy", "core_yoy", "dipi_yoy"]]
padding = 0.6
ax.set_ylim(all_vals.min().min() - padding, all_vals.max().max() + padding)

plt.savefig(
    "../graphics/exhibits/exhibit_04b_headline_core_dipi.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()